In [ ]:
from IPython.display import Markdown, display

display(Markdown("""
# Tsunami Evacuation Environment Workflow

This notebook builds the first GIS environment for tsunami evacuation modeling from the local road network, TAZ layer, and the scanned tsunami hazard map.
"""))

In [ ]:
from pathlib import Path
import os

import cv2
import geopandas as gpd
import numpy as np
import pandas as pd
from IPython.display import Image as NotebookImage, display
from PIL import Image
from shapely.geometry import LineString, Polygon

display(pd.DataFrame({"package": ["geopandas", "numpy", "pandas", "opencv"], "status": ["loaded"] * 4}))

In [ ]:
from pathlib import Path

# עובד ללא תלות בשם המשתמש, בתנאי שהתיקייה נקראת Documents\tzunami
PROJECT_ROOT = Path.home() / "Documents" / "tzunami"

MAPS_DIR = PROJECT_ROOT / "maps_key" / "maps_key"
TAZ_DIR = PROJECT_ROOT / "TAZ_v4.22_ITM" / "TAZ_v4.22_ITM"
BASE_2040_DIR = PROJECT_ROOT / "Base_2040"

IMAGE_PATH = PROJECT_ROOT / "tel aviv tsunami event.jpg"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".matplotlib"))
os.environ.setdefault("MPLBACKEND", "module://matplotlib_inline.backend_inline")
(PROJECT_ROOT / ".matplotlib").mkdir(exist_ok=True)

import matplotlib.pyplot as plt

display(pd.DataFrame({"setting": ["MPLCONFIGDIR", "MPLBACKEND"], "value": [os.environ["MPLCONFIGDIR"], os.environ["MPLBACKEND"]]}))

In [ ]:
required_paths = {
    "Roads": MAPS_DIR / "section.shp",
    "Nodes": MAPS_DIR / "nodes.shp",
    "TAZ": TAZ_DIR / "TAZ_v4.22_ITM.shp",
    "Hazard image": IMAGE_PATH,
    "Demand file": BASE_2040_DIR / "Das_Basic_2040_Aimsun.csv",
}

display(pd.DataFrame({
    "name": required_paths.keys(),
    "path": required_paths.values(),
    "exists": [path.exists() for path in required_paths.values()],
}))

In [ ]:
EPSG_ITM = 2039

frame = {
    "left": 92,
    "right": 950,
    "top": 33,
    "bottom": 1999,
}

extent = {
    "xmin": 170_000.0,
    "xmax": 185_000.0,
    "ymin": 645_000.0,
    "ymax": 684_000.0,
}

line_simplify_m = 15.0

display(pd.DataFrame([frame | extent | {"line_simplify_m": line_simplify_m, "crs": f"EPSG:{EPSG_ITM}"}]))

In [ ]:
image = Image.open(IMAGE_PATH).convert("RGB")
image_array = np.array(image)

display(pd.DataFrame({"width_px": [image.width], "height_px": [image.height], "mode": [image.mode]}))
display(image.resize((250, int(250 * image.height / image.width))))

In [ ]:
red_mask = (
    (image_array[:, :, 0] > 165)
    & (image_array[:, :, 1] < 120)
    & (image_array[:, :, 2] < 120)
    & ((image_array[:, :, 0] - image_array[:, :, 1]) > 70)
    & ((image_array[:, :, 0] - image_array[:, :, 2]) > 70)
).astype("uint8")

red_pixels = int(red_mask.sum())
ys, xs = np.where(red_mask == 1)

display(pd.DataFrame({
    "red_pixels": [red_pixels],
    "min_x_px": [int(xs.min())],
    "min_y_px": [int(ys.min())],
    "max_x_px": [int(xs.max())],
    "max_y_px": [int(ys.max())],
}))

In [ ]:
component_count, labels, stats, centroids = cv2.connectedComponentsWithStats(red_mask, 8)
component_rows = []
for label in range(1, component_count):
    x, y, width, height, area = stats[label]
    component_rows.append({
        "label": label,
        "area_px": int(area),
        "x_min_px": int(x),
        "y_min_px": int(y),
        "x_max_px": int(x + width - 1),
        "y_max_px": int(y + height - 1),
        "centroid_x_px": float(centroids[label][0]),
        "centroid_y_px": float(centroids[label][1]),
    })

components = pd.DataFrame(component_rows).sort_values("area_px", ascending=False)
largest_red_label = int(components.iloc[0]["label"])
boundary_mask = labels == largest_red_label

display(components.head(10))

In [ ]:
def pixel_to_itm(px: float, py: float) -> tuple[float, float]:
    x = extent["xmin"] + ((px - frame["left"]) / (frame["right"] - frame["left"])) * (extent["xmax"] - extent["xmin"])
    y = extent["ymax"] - ((py - frame["top"]) / (frame["bottom"] - frame["top"])) * (extent["ymax"] - extent["ymin"])
    return x, y


display(pd.DataFrame({
    "pixel": ["top-left", "bottom-right"],
    "itm": [pixel_to_itm(frame["left"], frame["top"]), pixel_to_itm(frame["right"], frame["bottom"])],
}))

In [ ]:
boundary_pixels = []
for py in range(frame["top"], frame["bottom"] + 1):
    row_xs = np.flatnonzero(boundary_mask[py, frame["left"] : frame["right"] + 1])
    if row_xs.size:
        boundary_pixels.append((frame["left"] + float(np.median(row_xs)), float(py)))

boundary_coords = [pixel_to_itm(px, py) for px, py in boundary_pixels]
hazard_boundary_line = LineString(boundary_coords).simplify(line_simplify_m, preserve_topology=False)

display(pd.DataFrame({"boundary_pixel_rows": [len(boundary_pixels)], "boundary_vertices": [len(hazard_boundary_line.coords)]}))

In [ ]:
line_coords = list(hazard_boundary_line.coords)
hazard_polygon = Polygon([
    (extent["xmin"], extent["ymax"]),
    line_coords[0],
    *line_coords[1:],
    (extent["xmin"], extent["ymin"]),
])

if not hazard_polygon.is_valid:
    hazard_polygon = hazard_polygon.buffer(0)

hazard_boundary = gpd.GeoDataFrame({"source": ["local_jpg_red_boundary"]}, geometry=[hazard_boundary_line], crs=f"EPSG:{EPSG_ITM}")
hazard_area = gpd.GeoDataFrame({"source": ["local_jpg_west_of_red_boundary"]}, geometry=[hazard_polygon], crs=f"EPSG:{EPSG_ITM}")

display(pd.DataFrame({"hazard_area_sq_km": [hazard_polygon.area / 1_000_000], "is_valid": [hazard_polygon.is_valid]}))
display(hazard_area)

In [ ]:
def read_layer(path: Path) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(path)

    if gdf.crs is None:
        raise ValueError(f"{path} has no CRS")

    return gdf.to_crs(EPSG_ITM)


roads = read_layer(MAPS_DIR / "section.shp")
nodes = read_layer(MAPS_DIR / "nodes.shp")
taz = read_layer(TAZ_DIR / "TAZ_v4.22_ITM.shp")

In [ ]:
roads_hazard = roads[roads.geometry.intersects(hazard_polygon)].copy()
roads_clipped = gpd.clip(roads, hazard_area).reset_index(drop=True)
nodes_hazard = nodes[nodes.geometry.within(hazard_polygon)].copy()
taz_hazard = taz[taz.geometry.intersects(hazard_polygon)].copy()

for layer in [roads_hazard, roads_clipped, nodes_hazard, taz_hazard]:
    layer["in_tsunami_hazard"] = True

hazard_counts = pd.DataFrame({
    "layer": ["road_sections_hazard", "road_sections_clipped", "nodes_hazard", "taz_hazard"],
    "rows": [len(roads_hazard), len(roads_clipped), len(nodes_hazard), len(taz_hazard)],
})

display(hazard_counts)

In [ ]:
sample_columns = [column for column in ["id", "eid", "name", "layer", "in_tsunami_hazard"] if column in roads_hazard.columns]

display(roads_hazard[sample_columns + ["geometry"]].head())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 12))
taz.boundary.plot(ax=ax, color="#c9c9c9", linewidth=0.25)
roads.plot(ax=ax, color="#737373", linewidth=0.25, alpha=0.45)
hazard_area.plot(ax=ax, facecolor="#ef4444", edgecolor="none", alpha=0.18)
roads_hazard.plot(ax=ax, color="#f97316", linewidth=0.65, alpha=0.95)
hazard_boundary.plot(ax=ax, color="#dc2626", linewidth=1.8)
ax.set_xlim(extent["xmin"], extent["xmax"])
ax.set_ylim(extent["ymin"], extent["ymax"])
ax.set_aspect("equal")
ax.set_title("Tsunami hazard environment, Tel Aviv area")
ax.set_xlabel("ITM Easting, EPSG:2039")
ax.set_ylabel("ITM Northing, EPSG:2039")
clean_preview_path = OUTPUT_DIR / "tsunami_environment_preview.png"
fig.tight_layout()
fig.savefig(clean_preview_path, dpi=180)

display(fig)
display(pd.DataFrame({"saved_preview": [clean_preview_path]}))
plt.close(fig)

In [ ]:
cropped_image = image_array[frame["top"] : frame["bottom"] + 1, frame["left"] : frame["right"] + 1]

fig, ax = plt.subplots(figsize=(8, 12))
ax.imshow(
    cropped_image,
    extent=(extent["xmin"], extent["xmax"], extent["ymin"], extent["ymax"]),
    origin="upper",
    alpha=0.86,
)
roads_hazard.plot(ax=ax, color="#0f172a", linewidth=0.75, alpha=0.9)
hazard_boundary.plot(ax=ax, color="#ef4444", linewidth=1.8)
ax.set_xlim(extent["xmin"], extent["xmax"])
ax.set_ylim(extent["ymin"], extent["ymax"])
ax.set_aspect("equal")
ax.set_title("Scanned tsunami map with extracted hazard roads")
ax.set_xlabel("ITM Easting, EPSG:2039")
ax.set_ylabel("ITM Northing, EPSG:2039")
raster_preview_path = OUTPUT_DIR / "tsunami_raster_overlay_preview.png"
fig.tight_layout()
fig.savefig(raster_preview_path, dpi=180)

display(fig)
display(pd.DataFrame({"saved_preview": [raster_preview_path]}))
plt.close(fig)

In [ ]:
gpkg_path = OUTPUT_DIR / "tsunami_environment.gpkg"
if gpkg_path.exists():
    gpkg_path.unlink()

layers_to_write = {
    "hazard_boundary": hazard_boundary,
    "hazard_area": hazard_area,
    "road_sections_hazard": roads_hazard,
    "road_sections_clipped": roads_clipped,
    "nodes_hazard": nodes_hazard,
    "taz_hazard": taz_hazard,
}

for layer_name, layer in layers_to_write.items():
    layer.to_file(gpkg_path, layer=layer_name, driver="GPKG")

display(pd.DataFrame({"layer": list(layers_to_write), "rows": [len(layer) for layer in layers_to_write.values()]}))
display(pd.DataFrame({"gpkg_path": [gpkg_path], "exists": [gpkg_path.exists()]}))

In [ ]:
summary = pd.DataFrame([
    ("maps_dir", str(MAPS_DIR)),
    ("taz_dir", str(TAZ_DIR)),
    ("base_2040_dir", str(BASE_2040_DIR)),
    ("image", str(IMAGE_PATH)),
    ("hazard_area_sq_km", hazard_polygon.area / 1_000_000),
    ("road_sections_total", len(roads)),
    ("road_sections_hazard", len(roads_hazard)),
    ("road_sections_clipped", len(roads_clipped)),
    ("nodes_total", len(nodes)),
    ("nodes_hazard", len(nodes_hazard)),
    ("taz_total", len(taz)),
    ("taz_hazard", len(taz_hazard)),
    ("gpkg", str(gpkg_path)),
    ("clean_preview", str(clean_preview_path)),
    ("raster_preview", str(raster_preview_path)),
], columns=["metric", "value"])

summary_path = OUTPUT_DIR / "tsunami_environment_summary.csv"
summary_status = "written"

try:
    summary.to_csv(summary_path, index=False)
except PermissionError:
    summary_path = OUTPUT_DIR / "tsunami_environment_summary_notebook.csv"
    summary.to_csv(summary_path, index=False)
    summary_status = "fallback_written_because_default_csv_was_locked"

display(summary)

display(pd.DataFrame({
    "summary_path": [summary_path],
    "status": [summary_status],
}))

In [ ]:
display(Markdown("""
# Mapping height evacuation options

Using TLV open maps
"""))

In [ ]:
from pathlib import Path

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

map1_path = OUTPUT_DIR / "tel_aviv_hazard_map.png"
map2_path = OUTPUT_DIR / "tel_aviv_hazard_with_refuge_buildings.png"

print(map1_path)
print(map2_path)

In [ ]:
buildings_path = PROJECT_ROOT / "tel_aviv_buildings.gpkg"

if not buildings_path.exists():
    raise FileNotFoundError(
        f"Building file was not found: {buildings_path}"
    )

buildings = gpd.read_file(buildings_path, layer="buildings")

if buildings.crs != f"EPSG:{EPSG_ITM}":
    buildings = buildings.to_crs(EPSG_ITM)

print("מספר מבנים שנטענו:", len(buildings))
print(buildings.columns)

display(buildings.head())

In [ ]:
buildings = buildings.copy()

# המרה לערכים מספריים
buildings["ms_komot"] = pd.to_numeric(buildings["ms_komot"], errors="coerce")
buildings["gova_simplex_2019"] = pd.to_numeric(buildings["gova_simplex_2019"], errors="coerce")
buildings["min_height"] = pd.to_numeric(buildings["min_height"], errors="coerce")
buildings["max_height"] = pd.to_numeric(buildings["max_height"], errors="coerce")

# אם יש גובה גג וגובה תחתית, אפשר לחשב גובה מבנה
buildings["height_from_roof_base"] = buildings["max_height"] - buildings["min_height"]

# גובה מועדף:
# קודם gova_simplex_2019, ואם חסר אז max_height - min_height
buildings["height_m"] = buildings["gova_simplex_2019"].fillna(
    buildings["height_from_roof_base"]
)

# safe_buildings = buildings[
#     (buildings["ms_komot"] >= 3) |
#     # (buildings["height_m"] >= 15)
# ].copy()
# safe_buildings = buildings[(buildings["ms_komot"] >= 4)&(buildings["height_m"] >= 15)].copy()
MIN_TOTAL_HEIGHT_M = 15.0

safe_buildings = buildings[
    (buildings["ms_komot"] >= 4)
    & (buildings["height_m"] >= MIN_TOTAL_HEIGHT_M)
].copy()


print("מבנים מספיק גבוהים:", len(safe_buildings))
# display(safe_buildings[["id_binyan", "t_sug_mivne", "ms_komot", "height_m"]].head())
display(safe_buildings[["id_binyan", "t_sug_mivne", "ms_komot", "height_m"]].head())

In [ ]:
hazard_union = hazard_area.unary_union

vertical_refuge_buildings = safe_buildings[
    safe_buildings.geometry.intersects(hazard_union)
].copy()

vertical_refuge_buildings["vertical_refuge"] = True

print("מבני מפלט פוטנציאליים בתוך אזור הסכנה:", len(vertical_refuge_buildings))

display(
    vertical_refuge_buildings[
        ["id_binyan", "t_sug_mivne", "shem_mivne", "ms_komot", "height_m"]
    ].head(20)
)

In [ ]:
from shapely.geometry import box
import geopandas as gpd

# =========================
# חלון תצוגה חדש
# =========================
ymin_view = 655000
ymax_view = 675000
xmax_view = 185000

# הקצה המערבי - אפשר לשנות ידנית אם תרצה
xmin_view = min(roads.total_bounds[0], hazard_area.total_bounds[0])

tel_aviv_window = box(xmin_view, ymin_view, xmax_view, ymax_view)

tel_aviv_window_gdf = gpd.GeoDataFrame(
    {"name": ["tel_aviv_window"]},
    geometry=[tel_aviv_window],
    crs=hazard_area.crs
)

print(xmin_view, xmax_view, ymin_view, ymax_view)

In [ ]:
roads_ta = gpd.clip(roads, tel_aviv_window_gdf)
hazard_area_ta = gpd.clip(hazard_area, tel_aviv_window_gdf)
taz_ta = gpd.clip(taz, tel_aviv_window_gdf)

print("roads_ta:", len(roads_ta))
print("hazard_area_ta:", len(hazard_area_ta))
print("taz_ta:", len(taz_ta))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 12))

# הים = רקע הצירים
ax.set_facecolor("lightblue")

# היבשה = TAZ
taz_ta.plot(ax=ax, facecolor="whitesmoke", edgecolor="lightgray", linewidth=0.3, alpha=1.0)

# כבישים
roads_ta.plot(ax=ax, linewidth=0.3, color="gray", alpha=0.7)

# אזור הצפה
hazard_area_ta.plot(ax=ax, color="red", alpha=0.25, edgecolor="darkred", linewidth=0.8)

ax.set_title("Tel Aviv Area - Tsunami Hazard Zone")
ax.set_xlabel("ITM X")
ax.set_ylabel("ITM Y")
ax.set_xlim(xmin_view, xmax_view)
ax.set_ylim(ymin_view, ymax_view)
ax.axis("equal")

fig.savefig(map1_path, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
vertical_refuge_ta = gpd.clip(vertical_refuge_buildings, tel_aviv_window_gdf)

print("vertical_refuge_ta:", len(vertical_refuge_ta))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))

# הים
ax.set_facecolor("lightblue")

# היבשה
taz_ta.plot(ax=ax, facecolor="whitesmoke", edgecolor="lightgray", linewidth=0.3, alpha=1.0)

# כבישים
roads_ta.plot(ax=ax, linewidth=0.25, color="gray", alpha=0.5)

# אזור הצפה
hazard_area_ta.plot(ax=ax, color="red", alpha=0.20, edgecolor="darkred", linewidth=0.8)

# מבני מפלט אפשריים
vertical_refuge_ta.plot(ax=ax, color="blue", alpha=0.85, edgecolor="navy", linewidth=0.4)

ax.set_title("Tel Aviv - Tsunami Hazard Zone with Potential Refuge Buildings")
ax.set_xlabel("ITM X")
ax.set_ylabel("ITM Y")
ax.set_xlim(xmin_view, xmax_view)
ax.set_ylim(ymin_view, ymax_view)
ax.axis("equal")

fig.savefig(map2_path, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from IPython.display import Markdown, display


display(Markdown("""
# Vertical refuge capacity model

Candidates have already passed the initial screen of at least 4 floors and a total height of at least 15 m. The capacity model uses only floors 4 and above, with 40% usable floor area.
"""))


In [ ]:
CAPACITY_MIN_SAFE_FLOOR = 4
USABLE_FACTOR = 0.4
M2_PER_PERSON = 1.0

capacity_map_path = OUTPUT_DIR / "tel_aviv_refuge_capacity_map.png"
capacity_gpkg_path = OUTPUT_DIR / "vertical_refuge_capacity.gpkg"
capacity_csv_path = OUTPUT_DIR / "vertical_refuge_capacity_summary.csv"

capacity_assumptions = pd.DataFrame([
    ("candidate screen", "At least 4 floors and total height of at least 15 m"),
    ("safe floors", "Only floors 4 and above"),
    ("usable factor", USABLE_FACTOR),
    ("m2 per person", M2_PER_PERSON),
    ("capacity formula", "footprint_m2 * safe_floors * usable_factor / m2_per_person"),
], columns=["assumption", "value"])

display(capacity_assumptions)


In [ ]:
vertical_refuge_capacity = vertical_refuge_buildings.copy()

if vertical_refuge_capacity.crs != f"EPSG:{EPSG_ITM}":
    vertical_refuge_capacity = vertical_refuge_capacity.to_crs(EPSG_ITM)

vertical_refuge_capacity["floors_total"] = pd.to_numeric(
    vertical_refuge_capacity["ms_komot"], errors="coerce"
)
vertical_refuge_capacity = vertical_refuge_capacity[
    vertical_refuge_capacity["floors_total"] >= CAPACITY_MIN_SAFE_FLOOR
].copy()

# Using only floors 4 and above: 4 floors -> 1 usable refuge floor, 5 floors -> 2, etc.
vertical_refuge_capacity["safe_floors"] = (
    vertical_refuge_capacity["floors_total"] - (CAPACITY_MIN_SAFE_FLOOR - 1)
).clip(lower=0)

vertical_refuge_capacity["footprint_m2"] = vertical_refuge_capacity.geometry.area
vertical_refuge_capacity["usable_factor"] = USABLE_FACTOR
vertical_refuge_capacity["m2_per_person"] = M2_PER_PERSON
vertical_refuge_capacity["usable_refuge_m2"] = (
    vertical_refuge_capacity["footprint_m2"]
    * vertical_refuge_capacity["safe_floors"]
    * vertical_refuge_capacity["usable_factor"]
)
vertical_refuge_capacity["capacity_people"] = np.floor(
    vertical_refuge_capacity["usable_refuge_m2"] / vertical_refuge_capacity["m2_per_person"]
).astype("int64")

vertical_refuge_capacity["capacity_class"] = pd.cut(
    vertical_refuge_capacity["capacity_people"],
    bins=[-1, 99, 499, 1499, np.inf],
    labels=["small (<100)", "medium (100-499)", "large (500-1499)", "major (1500+)"],
).astype(str)

capacity_columns = [
    column for column in ["id_binyan", "t_sug_mivne", "shem_mivne"]
    if column in vertical_refuge_capacity.columns
] + [
    "floors_total", "safe_floors", "footprint_m2", "usable_refuge_m2", "capacity_people", "capacity_class"
]

capacity_top20 = vertical_refuge_capacity[capacity_columns].sort_values(
    "capacity_people", ascending=False
).head(20)

display(capacity_top20)


In [ ]:
capacity_summary = pd.DataFrame([
    ("refuge_buildings", len(vertical_refuge_capacity)),
    ("total_capacity_people", int(vertical_refuge_capacity["capacity_people"].sum())),
    ("median_capacity_people", float(vertical_refuge_capacity["capacity_people"].median())),
    ("mean_capacity_people", float(vertical_refuge_capacity["capacity_people"].mean())),
    ("max_capacity_people", int(vertical_refuge_capacity["capacity_people"].max())),
    ("total_usable_refuge_m2", float(vertical_refuge_capacity["usable_refuge_m2"].sum())),
], columns=["metric", "value"])

capacity_by_class = (
    vertical_refuge_capacity.groupby("capacity_class", dropna=False)
    .agg(
        buildings=("capacity_people", "size"),
        total_capacity_people=("capacity_people", "sum"),
        median_capacity_people=("capacity_people", "median"),
    )
    .reset_index()
)

display(capacity_summary)
display(capacity_by_class)


In [ ]:
def write_gpkg_with_fallback(gdf, path, layer_name):
    target = path
    status = "written"
    try:
        if target.exists():
            target.unlink()
        gdf.to_file(target, layer=layer_name, driver="GPKG")
    except PermissionError:
        target = path.with_name(path.stem + "_notebook" + path.suffix)
        if target.exists():
            target.unlink()
        gdf.to_file(target, layer=layer_name, driver="GPKG")
        status = "fallback_written_because_default_file_was_locked"
    return target, status


def write_csv_with_fallback(df, path):
    target = path
    status = "written"
    try:
        df.to_csv(target, index=False)
    except PermissionError:
        target = path.with_name(path.stem + "_notebook" + path.suffix)
        df.to_csv(target, index=False)
        status = "fallback_written_because_default_file_was_locked"
    return target, status

capacity_gpkg_written, gpkg_status = write_gpkg_with_fallback(
    vertical_refuge_capacity,
    capacity_gpkg_path,
    "vertical_refuge_capacity",
)
capacity_csv_written, csv_status = write_csv_with_fallback(
    capacity_by_class,
    capacity_csv_path,
)

capacity_outputs = pd.DataFrame([
    ("capacity_gpkg", capacity_gpkg_written, gpkg_status),
    ("capacity_csv", capacity_csv_written, csv_status),
], columns=["output", "path", "status"])

display(capacity_outputs)


In [ ]:
vertical_refuge_capacity_ta = gpd.clip(vertical_refuge_capacity, tel_aviv_window_gdf)

capacity_colors = {
    "small (<100)": "#93c5fd",
    "medium (100-499)": "#2563eb",
    "large (500-1499)": "#7c3aed",
    "major (1500+)": "#f59e0b",
}

fig, ax = plt.subplots(figsize=(10, 12))
ax.set_facecolor("lightblue")

taz_ta.plot(ax=ax, facecolor="whitesmoke", edgecolor="lightgray", linewidth=0.3, alpha=1.0)
roads_ta.plot(ax=ax, linewidth=0.25, color="gray", alpha=0.45)
hazard_area_ta.plot(ax=ax, color="red", alpha=0.18, edgecolor="darkred", linewidth=0.8)

for capacity_class, group in vertical_refuge_capacity_ta.groupby("capacity_class"):
    group.plot(
        ax=ax,
        color=capacity_colors.get(capacity_class, "#64748b"),
        edgecolor="black",
        linewidth=0.25,
        alpha=0.90,
        label=capacity_class,
    )

ax.set_title("Tel Aviv - Vertical Refuge Building Capacity")
ax.set_xlabel("ITM X")
ax.set_ylabel("ITM Y")
# ax.set_xlim(xmin_view, xmax_view)
# ax.set_ylim(ymin_view, ymax_view)
ax.set_xlim(175000, 182000)
ax.set_ylim(660000, 668000)
# ax.axis("equal")
ax.set_aspect("equal", adjustable="box")
ax.legend(title="Estimated capacity", loc="upper right")

fig.savefig(capacity_map_path, dpi=300, bbox_inches="tight")
plt.show()

display(pd.DataFrame({"capacity_map_path": [capacity_map_path], "mapped_buildings": [len(vertical_refuge_capacity_ta)]}))


In [ ]:
from matplotlib.patches import Patch

# =========================
# גבולות האזור המצומצם להצגה
# =========================
xmin_zoom = 175000
xmax_zoom = 182000
ymin_zoom = 660000
ymax_zoom = 668000

zoom_window = box(xmin_zoom, ymin_zoom, xmax_zoom, ymax_zoom)

zoom_window_gdf = gpd.GeoDataFrame(
    {"name": ["zoom_window"]},
    geometry=[zoom_window],
    crs=vertical_refuge_capacity.crs
)

# =========================
# חיתוך כל השכבות לאזור המצומצם
# =========================
taz_zoom = gpd.clip(taz_ta, zoom_window_gdf)
roads_zoom = gpd.clip(roads_ta, zoom_window_gdf)
hazard_area_zoom = gpd.clip(hazard_area_ta, zoom_window_gdf)
vertical_refuge_capacity_zoom = gpd.clip(vertical_refuge_capacity, zoom_window_gdf)

# =========================
# צבעים לפי קיבולת
# =========================
capacity_colors = {
    "small (<100)": "#93c5fd",
    "medium (100-499)": "#2563eb",
    "large (500-1499)": "#7c3aed",
    "major (1500+)": "#f59e0b",
}

# =========================
# ציור המפה
# =========================
fig, ax = plt.subplots(figsize=(10, 12))
ax.set_facecolor("lightblue")

# יבשה / אזורי TAZ
taz_zoom.plot(
    ax=ax,
    facecolor="whitesmoke",
    edgecolor="lightgray",
    linewidth=0.3,
    alpha=1.0,
    zorder=1
)

# כבישים
roads_zoom.plot(
    ax=ax,
    linewidth=0.25,
    color="gray",
    alpha=0.45,
    zorder=2
)

# אזור הצפה
hazard_area_zoom.plot(
    ax=ax,
    color="red",
    alpha=0.18,
    edgecolor="darkred",
    linewidth=0.8,
    zorder=3
)

# מבני מפלט לפי קיבולת
for capacity_class, group in vertical_refuge_capacity_zoom.groupby("capacity_class"):
    group.plot(
        ax=ax,
        color=capacity_colors.get(capacity_class, "#64748b"),
        edgecolor="black",
        linewidth=0.25,
        alpha=0.90,
        zorder=4
    )

# =========================
# מקרא ידני
# =========================
existing_classes = vertical_refuge_capacity_zoom["capacity_class"].dropna().unique()

legend_handles = [
    Patch(
        facecolor=color,
        edgecolor="black",
        label=capacity_class
    )
    for capacity_class, color in capacity_colors.items()
    if capacity_class in existing_classes
]

ax.legend(
    handles=legend_handles,
    title="Estimated capacity",
    loc="upper left"
)

# =========================
# הגדרות תצוגה
# =========================
ax.set_title("Tel Aviv - Vertical Refuge Building Capacity")
ax.set_xlabel("ITM X")
ax.set_ylabel("ITM Y")

ax.set_xlim(xmin_zoom, xmax_zoom)
ax.set_ylim(ymin_zoom, ymax_zoom)
ax.set_aspect("equal", adjustable="box")

# =========================
# שמירה והצגה
# =========================
fig.savefig(capacity_map_path, dpi=300, bbox_inches="tight")
plt.show()

display(pd.DataFrame({
    "capacity_map_path": [capacity_map_path],
    "mapped_buildings": [len(vertical_refuge_capacity_zoom)]
}))

In [ ]:
from IPython.display import Markdown, display


display(Markdown("""
# Refuge suitability and access classification

This section keeps only buildings that already passed the previous vertical-refuge conditions, then classifies whether they are realistic planned refuges or only low-confidence spontaneous options.
"""))


In [ ]:
suitability_gpkg_path = OUTPUT_DIR / "vertical_refuge_suitability.gpkg"
suitability_csv_path = OUTPUT_DIR / "vertical_refuge_suitability_summary.csv"
suitability_map_path = OUTPUT_DIR / "tel_aviv_refuge_suitability_map.png"

# All buildings here already satisfy the previous conditions: inside the hazard area, 4+ floors,
# safe floors are floors 4 and above, and capacity was computed using usable_factor=0.4.
vertical_refuge_suitability = vertical_refuge_capacity.copy()

vertical_refuge_suitability["building_type_raw"] = vertical_refuge_suitability["t_sug_mivne"].fillna("missing").astype(str)
vertical_refuge_suitability["building_name_raw"] = vertical_refuge_suitability["shem_mivne"].fillna("").astype(str).str.strip()
vertical_refuge_suitability["has_building_name"] = vertical_refuge_suitability["building_name_raw"].ne("")

classification_scope = pd.DataFrame([
    ("input_layer", "vertical_refuge_capacity"),
    ("minimum_total_floors", CAPACITY_MIN_SAFE_FLOOR),
    ("safe_floor_rule", "only floors 4 and above"),
    ("usable_factor", USABLE_FACTOR),
    ("candidate_buildings", len(vertical_refuge_suitability)),
], columns=["item", "value"])

display(classification_scope)


In [ ]:
def contains_any_text(series, words):
    mask = pd.Series(False, index=series.index)
    for word in words:
        mask |= series.str.contains(word, case=False, regex=False, na=False)
    return mask

name_text = vertical_refuge_suitability["building_name_raw"]
type_text = vertical_refuge_suitability["building_type_raw"]

is_public_type = type_text.eq("מבנה ציבור")
is_construction_or_temporary = type_text.isin(["מבנה בבנייה", "מבנה ארעי"])
is_missing_or_unknown_type = type_text.isin(["missing", "לא ידוע", "", "nan"])

is_hotel = contains_any_text(name_text, [
    "מלון", "hotel", "הילטון", "שרתון", "הרודס", "קראון", "פלאזה",
    "אורכידאה", "קמפינסקי", "רנסנס", "קרלטון", "דן פנורמה", "דן תל אביב"
])
is_parking = contains_any_text(name_text, ["חניון", "parking"])
is_business_tower = contains_any_text(name_text, [
    "מגדל", "טאואר", "tower", "טרייד", "תעשינים", "טקסטיל", "גיבור", "סיטי", "מכבי"
])
is_school_or_community = contains_any_text(name_text, [
    "בית ספר", "גן ילדים", "מרכז קהילתי", "מועדון", "מתנ", "אוניברסיטה", "מכללה", "חינוכי"
])
is_medical = contains_any_text(name_text, ["רפואי", "מרפאה", "בית חולים"])

conditions = [
    is_construction_or_temporary,
    is_missing_or_unknown_type,
    is_parking,
    is_public_type | is_school_or_community | is_medical,
    is_hotel,
    is_business_tower,
    vertical_refuge_suitability["has_building_name"],
]

classes = [
    "exclude_construction_or_temporary",
    "exclude_missing_or_unknown_type",
    "planned_parking_high",
    "planned_public_high",
    "planned_hotel_medium_high",
    "planned_business_medium",
    "spontaneous_named_low",
]

vertical_refuge_suitability["refuge_suitability_class"] = np.select(
    conditions,
    classes,
    default="spontaneous_regular_unknown_low",
)

access_probability_by_class = {
    "planned_public_high": 0.80,
    "planned_parking_high": 0.70,
    "planned_hotel_medium_high": 0.60,
    "planned_business_medium": 0.40,
    "spontaneous_named_low": 0.20,
    "spontaneous_regular_unknown_low": 0.05,
    "exclude_construction_or_temporary": 0.00,
    "exclude_missing_or_unknown_type": 0.00,
}

planned_classes = {
    "planned_public_high",
    "planned_parking_high",
    "planned_hotel_medium_high",
    "planned_business_medium",
}

vertical_refuge_suitability["access_probability"] = vertical_refuge_suitability["refuge_suitability_class"].map(access_probability_by_class).fillna(0.0)
vertical_refuge_suitability["refuge_assignment_eligible"] = vertical_refuge_suitability["refuge_suitability_class"].isin(planned_classes)
vertical_refuge_suitability["effective_capacity_people"] = np.floor(
    vertical_refuge_suitability["capacity_people"] * vertical_refuge_suitability["access_probability"]
).astype("int64")
vertical_refuge_suitability["planned_effective_capacity_people"] = np.where(
    vertical_refuge_suitability["refuge_assignment_eligible"],
    vertical_refuge_suitability["effective_capacity_people"],
    0,
).astype("int64")

vertical_refuge_suitability["suitability_reason"] = np.select(
    conditions,
    [
        "Excluded: construction or temporary structure",
        "Excluded: missing or unknown building type",
        "Named parking structure",
        "Public/community/education/medical signal",
        "Hotel signal in building name",
        "Business/tower signal in building name",
        "Named building, but no strong public-access signal",
    ],
    default="Regular unnamed building; likely private/residential/mixed, low confidence",
)

summary_cols = [
    "refuge_suitability_class", "access_probability", "refuge_assignment_eligible",
    "capacity_people", "effective_capacity_people", "planned_effective_capacity_people"
]

display(vertical_refuge_suitability[[
    "id_binyan", "t_sug_mivne", "shem_mivne", "floors_total", "capacity_people",
    "refuge_suitability_class", "access_probability", "effective_capacity_people",
    "refuge_assignment_eligible", "suitability_reason"
]].sort_values(["refuge_assignment_eligible", "effective_capacity_people"], ascending=[False, False]).head(30))


In [ ]:
refuge_suitability_summary = (
    vertical_refuge_suitability.groupby(["refuge_suitability_class", "refuge_assignment_eligible", "access_probability"], dropna=False)
    .agg(
        buildings=("id_binyan", "count"),
        theoretical_capacity_people=("capacity_people", "sum"),
        effective_capacity_people=("effective_capacity_people", "sum"),
        planned_effective_capacity_people=("planned_effective_capacity_people", "sum"),
        median_capacity_people=("capacity_people", "median"),
    )
    .reset_index()
    .sort_values("planned_effective_capacity_people", ascending=False)
)

overall_suitability_summary = pd.DataFrame([
    ("all_previous_condition_buildings", len(vertical_refuge_suitability)),
    ("planned_refuge_buildings", int(vertical_refuge_suitability["refuge_assignment_eligible"].sum())),
    ("spontaneous_or_excluded_buildings", int((~vertical_refuge_suitability["refuge_assignment_eligible"]).sum())),
    ("theoretical_capacity_people", int(vertical_refuge_suitability["capacity_people"].sum())),
    ("effective_capacity_people_all_classes", int(vertical_refuge_suitability["effective_capacity_people"].sum())),
    ("planned_effective_capacity_people", int(vertical_refuge_suitability["planned_effective_capacity_people"].sum())),
], columns=["metric", "value"])

display(overall_suitability_summary)
display(refuge_suitability_summary)


In [ ]:
top_planned_refuges = vertical_refuge_suitability[
    vertical_refuge_suitability["refuge_assignment_eligible"]
].sort_values("planned_effective_capacity_people", ascending=False)

top_planned_columns = [
    "id_binyan", "t_sug_mivne", "shem_mivne", "floors_total", "capacity_people",
    "access_probability", "planned_effective_capacity_people", "refuge_suitability_class",
    "suitability_reason"
]

display(top_planned_refuges[top_planned_columns].head(50))


In [ ]:
def write_gpkg_with_fallback(gdf, path, layer_name):
    target = path
    status = "written"
    try:
        if target.exists():
            target.unlink()
        gdf.to_file(target, layer=layer_name, driver="GPKG")
    except PermissionError:
        target = path.with_name(path.stem + "_notebook" + path.suffix)
        if target.exists():
            target.unlink()
        gdf.to_file(target, layer=layer_name, driver="GPKG")
        status = "fallback_written_because_default_file_was_locked"
    return target, status

try:
    suitability_gpkg_written, suitability_gpkg_status = write_gpkg_with_fallback(
        vertical_refuge_suitability,
        suitability_gpkg_path,
        "vertical_refuge_suitability",
    )
except NameError:
    suitability_gpkg_written = suitability_gpkg_path
    suitability_gpkg_status = "not_written_missing_write_helper"

suitability_csv_written, suitability_csv_status = write_csv_with_fallback(
    refuge_suitability_summary,
    suitability_csv_path,
)

suitability_outputs = pd.DataFrame([
    ("suitability_gpkg", suitability_gpkg_written, suitability_gpkg_status),
    ("suitability_summary_csv", suitability_csv_written, suitability_csv_status),
], columns=["output", "path", "status"])

display(suitability_outputs)


In [ ]:
vertical_refuge_suitability_ta = gpd.clip(vertical_refuge_suitability, tel_aviv_window_gdf)

suitability_colors = {
    "planned_public_high": "#16a34a",
    "planned_parking_high": "#0f766e",
    "planned_hotel_medium_high": "#2563eb",
    "planned_business_medium": "#7c3aed",
    "spontaneous_named_low": "#f59e0b",
    "spontaneous_regular_unknown_low": "#94a3b8",
    "exclude_construction_or_temporary": "#ef4444",
    "exclude_missing_or_unknown_type": "#6b7280",
}

fig, ax = plt.subplots(figsize=(11, 12))
ax.set_facecolor("lightblue")

taz_ta.plot(ax=ax, facecolor="whitesmoke", edgecolor="lightgray", linewidth=0.3, alpha=1.0)
roads_ta.plot(ax=ax, linewidth=0.25, color="gray", alpha=0.4)
hazard_area_ta.plot(ax=ax, color="red", alpha=0.18, edgecolor="darkred", linewidth=0.8)

for suitability_class, group in vertical_refuge_suitability_ta.groupby("refuge_suitability_class"):
    group.plot(
        ax=ax,
        color=suitability_colors.get(suitability_class, "#64748b"),
        edgecolor="black",
        linewidth=0.25,
        alpha=0.9,
        label=suitability_class,
    )

ax.set_title("Tel Aviv - Vertical Refuge Suitability and Access Classification")
ax.set_xlabel("ITM X")
ax.set_ylabel("ITM Y")
ax.set_xlim(xmin_view, xmax_view)
ax.set_ylim(ymin_view, ymax_view)
ax.axis("equal")
ax.legend(title="Suitability class", loc="upper right", fontsize=8)

fig.savefig(suitability_map_path, dpi=300, bbox_inches="tight")
plt.show()

display(pd.DataFrame({
    "suitability_map_path": [suitability_map_path],
    "mapped_buildings": [len(vertical_refuge_suitability_ta)],
    "planned_mapped_buildings": [int(vertical_refuge_suitability_ta["refuge_assignment_eligible"].sum())],
}))


In [ ]:
from IPython.display import Markdown, display


display(Markdown("""
# Demand allocation to refuge buildings or outside flood area

This section estimates how many people are inside each flooded TAZ at the evacuation time, then assigns each demand zone either to the nearest vertical-refuge building or to the nearest point outside the flooded area.
"""))


In [ ]:
from shapely.geometry import LineString
from shapely.ops import nearest_points

# Scenario clock: decimal hours in the Daily Activity Schedule (DAS).
# This is the time when the tsunami alert is received, not a duration.
ALERT_TIME_HOUR = 11.0
# Temporary far-field scenario assumption; replace with the selected seismological scenario.
TSUNAMI_ARRIVAL_HOUR = 13.0

if TSUNAMI_ARRIVAL_HOUR <= ALERT_TIME_HOUR:
    raise ValueError("TSUNAMI_ARRIVAL_HOUR must be later than ALERT_TIME_HOUR")

WARNING_WINDOW_MIN = int(round((TSUNAMI_ARRIVAL_HOUR - ALERT_TIME_HOUR) * 60))
_alert_total_minutes = int(round(ALERT_TIME_HOUR * 60))
ALERT_TIME_TAG = f"{_alert_total_minutes // 60:02d}_{_alert_total_minutes % 60:02d}"
DEMAND_CHUNK_SIZE = 1_000_000
RECOMPUTE_DEMAND = False

DEMAND_FILE = BASE_2040_DIR / "Das_Basic_2040_Aimsun.csv"
# Full person-state snapshot: activity, initial location, and transit at the alert time.
demand_cache_path = OUTPUT_DIR / f"das_people_by_taz_alert_{ALERT_TIME_TAG}_person_states_v2.csv"
assignment_gpkg_path = OUTPUT_DIR / "evacuation_assignment.gpkg"
assignment_summary_csv_path = OUTPUT_DIR / "evacuation_assignment_summary.csv"
assignment_map_path = OUTPUT_DIR / "evacuation_assignment_map.png"

assignment_parameters = pd.DataFrame([
    ("alert_time_hour", ALERT_TIME_HOUR),
    ("tsunami_arrival_hour", TSUNAMI_ARRIVAL_HOUR),
    ("warning_window_min", WARNING_WINDOW_MIN),
    ("demand_definition", "DAS person-state snapshot: activity, initial location, and transit allocation"),
    ("demand source", DEMAND_FILE),
    ("demand cache", demand_cache_path),
    ("assignment distance", "shortest road-network path in meters, including access connectors"),
    ("destination options", "vertical refuge building or outside flooded area"),
], columns=["parameter", "value"])

display(assignment_parameters)


In [ ]:
# Build one population snapshot at the alert time.  Every represented person is
# classified as being at an activity, at the initial recorded location, or in transit.
if demand_cache_path.exists() and not RECOMPUTE_DEMAND:
    people_by_zone = pd.read_csv(demand_cache_path)
    demand_source_status = "loaded_from_cache"
else:
    usecols = [
        "person_id", "arrival_time", "departure_time", "prev_stop_departure_time",
        "stop_zone", "prev_stop_zone",
    ]
    activity_records = []
    transit_records = []
    first_record_candidates = []
    rows_read = 0

    for chunk in pd.read_csv(DEMAND_FILE, usecols=usecols, chunksize=DEMAND_CHUNK_SIZE):
        rows_read += len(chunk)
        for column in ["arrival_time", "departure_time", "prev_stop_departure_time", "stop_zone", "prev_stop_zone"]:
            chunk[column] = pd.to_numeric(chunk[column], errors="coerce")

        activity_records.append(chunk.loc[
            (chunk["arrival_time"] <= ALERT_TIME_HOUR)
            & (chunk["departure_time"] > ALERT_TIME_HOUR),
            ["person_id", "stop_zone"],
        ].copy())

        transit_records.append(chunk.loc[
            (chunk["prev_stop_departure_time"] <= ALERT_TIME_HOUR)
            & (chunk["arrival_time"] > ALERT_TIME_HOUR),
            ["person_id", "prev_stop_zone", "stop_zone", "prev_stop_departure_time", "arrival_time"],
        ].copy())

        # The CSV is processed in chunks, so keep the earliest candidate from every
        # chunk and resolve the true first record after concatenation.
        first_record_candidates.append(
            chunk.sort_values(["person_id", "prev_stop_departure_time"])
            .groupby("person_id", as_index=False)
            .first()[["person_id", "prev_stop_zone", "prev_stop_departure_time"]]
        )
        print(f"rows_read={rows_read:,}")

    at_activity = pd.concat(activity_records, ignore_index=True).drop_duplicates("person_id")
    in_transit = pd.concat(transit_records, ignore_index=True).drop_duplicates("person_id")
    first_records = (
        pd.concat(first_record_candidates, ignore_index=True)
        .sort_values(["person_id", "prev_stop_departure_time"])
        .drop_duplicates("person_id", keep="first")
    )

    activity_ids = set(at_activity["person_id"].dropna())
    transit_ids = set(in_transit["person_id"].dropna())
    overlapping_state_ids = activity_ids & transit_ids
    if overlapping_state_ids:
        # At a malformed boundary record, retain the activity state deterministically.
        in_transit = in_transit[~in_transit["person_id"].isin(overlapping_state_ids)].copy()
        transit_ids -= overlapping_state_ids

    at_initial_location = first_records[
        (first_records["prev_stop_departure_time"] > ALERT_TIME_HOUR)
        & ~first_records["person_id"].isin(activity_ids | transit_ids)
    ].copy()

    activity_zone_weights = at_activity.rename(columns={"stop_zone": "TAZV41"})
    activity_zone_weights["people_at_alert"] = 1.0
    activity_zone_weights["person_state"] = "at_activity"

    initial_zone_weights = at_initial_location.rename(columns={"prev_stop_zone": "TAZV41"})
    initial_zone_weights["people_at_alert"] = 1.0
    initial_zone_weights["person_state"] = "at_initial_location"

    transit_zone_weights = in_transit.copy()
    travel_duration = transit_zone_weights["arrival_time"] - transit_zone_weights["prev_stop_departure_time"]
    transit_zone_weights["destination_weight"] = np.where(
        travel_duration > 0,
        ((ALERT_TIME_HOUR - transit_zone_weights["prev_stop_departure_time"]) / travel_duration).clip(0, 1),
        0.5,
    )
    transit_zone_weights["origin_weight"] = 1.0 - transit_zone_weights["destination_weight"]

    transit_from_origin = transit_zone_weights[["person_id", "prev_stop_zone", "origin_weight"]].rename(
        columns={"prev_stop_zone": "TAZV41", "origin_weight": "people_at_alert"}
    )
    transit_from_destination = transit_zone_weights[["person_id", "stop_zone", "destination_weight"]].rename(
        columns={"stop_zone": "TAZV41", "destination_weight": "people_at_alert"}
    )
    transit_from_origin["person_state"] = "in_transit_estimated_origin_share"
    transit_from_destination["person_state"] = "in_transit_estimated_destination_share"

    person_zone_weights = pd.concat([
        activity_zone_weights[["person_id", "TAZV41", "people_at_alert", "person_state"]],
        initial_zone_weights[["person_id", "TAZV41", "people_at_alert", "person_state"]],
        transit_from_origin[["person_id", "TAZV41", "people_at_alert", "person_state"]],
        transit_from_destination[["person_id", "TAZV41", "people_at_alert", "person_state"]],
    ], ignore_index=True)
    person_zone_weights["TAZV41"] = pd.to_numeric(person_zone_weights["TAZV41"], errors="coerce")
    person_zone_weights = person_zone_weights.dropna(subset=["person_id", "TAZV41", "people_at_alert"])
    person_zone_weights["TAZV41"] = person_zone_weights["TAZV41"].astype("int64")
    person_zone_weights = person_zone_weights[person_zone_weights["people_at_alert"] > 0].copy()

    people_by_zone = (
        person_zone_weights.pivot_table(
            index="TAZV41",
            columns="person_state",
            values="people_at_alert",
            aggfunc="sum",
            fill_value=0.0,
        )
        .reset_index()
    )
    people_by_zone["people_at_alert"] = people_by_zone.drop(columns="TAZV41").sum(axis=1)
    people_by_zone.to_csv(demand_cache_path, index=False)
    demand_source_status = "computed_from_das_csv"

people_by_zone["TAZV41"] = pd.to_numeric(people_by_zone["TAZV41"], errors="coerce").astype("Int64")
state_columns = [column for column in people_by_zone.columns if column != "TAZV41"]
for column in state_columns:
    people_by_zone[column] = pd.to_numeric(people_by_zone[column], errors="coerce").fillna(0.0)

active_people_by_zone = people_by_zone  # Backward-compatible name for downstream exploration cells.
active_people_summary = pd.DataFrame([
    ("status", demand_source_status),
    ("zones_with_people_at_alert", int(people_by_zone["TAZV41"].nunique())),
    ("people_at_alert_total", round(float(people_by_zone["people_at_alert"].sum()), 2)),
    ("at_activity_total", round(float(people_by_zone.get("at_activity", pd.Series(dtype=float)).sum()), 2)),
    ("at_initial_location_total", round(float(people_by_zone.get("at_initial_location", pd.Series(dtype=float)).sum()), 2)),
    ("in_transit_total", round(float(
        people_by_zone.get("in_transit_estimated_origin_share", pd.Series(dtype=float)).sum()
        + people_by_zone.get("in_transit_estimated_destination_share", pd.Series(dtype=float)).sum()
    ), 2)),
], columns=["metric", "value"])

display(active_people_summary)
display(people_by_zone.sort_values("people_at_alert", ascending=False).head(10))


In [ ]:
from shapely.geometry import Point

taz_demand = taz.copy()
if taz_demand.crs != f"EPSG:{EPSG_ITM}":
    taz_demand = taz_demand.to_crs(EPSG_ITM)

taz_demand["TAZV41"] = pd.to_numeric(
    taz_demand["TAZV41"],
    errors="coerce",
).astype("Int64")

taz_demand = taz_demand.merge(people_by_zone, on="TAZV41", how="left")

demand_columns = [
    column for column in people_by_zone.columns
    if column != "TAZV41"
]

for column in demand_columns:
    taz_demand[column] = pd.to_numeric(
        taz_demand[column],
        errors="coerce",
    ).fillna(0.0)

hazard_union = hazard_area.union_all()
flooded_geometry = taz_demand.geometry.intersection(hazard_union)

taz_demand["taz_area_m2"] = taz_demand.geometry.area
taz_demand["flooded_area_m2"] = flooded_geometry.area
taz_demand["flooded_share"] = (
    taz_demand["flooded_area_m2"] / taz_demand["taz_area_m2"]
).clip(lower=0, upper=1)

# Demand may be fractional because in-transit people are split between TAZs.
taz_demand["flooded_people"] = (
    taz_demand["people_at_alert"] * taz_demand["flooded_share"]
)

demand_polygons = taz_demand[
    (taz_demand["flooded_area_m2"] > 0)
    & (taz_demand["flooded_people"] > 0)
].copy()

demand_polygons["geometry"] = flooded_geometry.loc[demand_polygons.index]
demand_polygons = gpd.GeoDataFrame(
    demand_polygons,
    geometry="geometry",
    crs=taz_demand.crs,
)

# ==========================================================
# Split each flooded TAZ into several spatial demand points
# ==========================================================
DEMAND_PEOPLE_PER_POINT = 250.0
MAX_DEMAND_POINTS_PER_TAZ = 20
DEMAND_POINT_SEED = 20260819

# Keep random points close to the represented road system when possible.
# This only relocates an implausible random sample; demand totals are unchanged.
DEMAND_POINT_ROAD_ACCESS_TARGET_M = 500.0
DEMAND_POINT_ROAD_RESAMPLE_ATTEMPTS = 500

rng = np.random.default_rng(DEMAND_POINT_SEED)


def random_point_inside(geometry, random_generator, max_attempts=500):
    """Draw one reproducible random point inside a polygon or multipolygon."""
    minx, miny, maxx, maxy = geometry.bounds

    for _ in range(max_attempts):
        point = Point(
            random_generator.uniform(minx, maxx),
            random_generator.uniform(miny, maxy),
        )
        if geometry.contains(point):
            return point

    # Safe fallback for narrow or complex polygons.
    return geometry.representative_point()


origin_records = []

for _, polygon_row in demand_polygons.iterrows():
    total_flooded_people = float(polygon_row["flooded_people"])

    point_count = int(np.ceil(
        total_flooded_people / DEMAND_PEOPLE_PER_POINT
    ))
    point_count = max(1, min(point_count, MAX_DEMAND_POINTS_PER_TAZ))

    people_per_point = total_flooded_people / point_count

    for point_index in range(point_count):
        record = polygon_row.drop(labels="geometry").to_dict()

        # Distribute the estimated flooded population among the new points.
        for column in demand_columns:
            record[column] = (
                float(polygon_row[column])
                * float(polygon_row["flooded_share"])
                / point_count
            )

        record["parent_flooded_people"] = total_flooded_people
        record["flooded_people"] = people_per_point
        record["demand_point_index"] = point_index + 1
        record["demand_point_count"] = point_count
        record["origin_id"] = (
            f"TAZ_{int(polygon_row['TAZV41'])}_P_{point_index + 1:02d}"
        )
        record["geometry"] = random_point_inside(
            polygon_row.geometry,
            rng,
        )

        origin_records.append(record)

demand_origins = gpd.GeoDataFrame(
    origin_records,
    geometry="geometry",
    crs=taz_demand.crs,
)

if len(demand_origins) == 0:
    raise ValueError("No demand origins were created inside the flooded area.")

# Most locations are sampled uniformly in the flooded part of their TAZ.
# If one happens to fall far from every road, draw candidates in the same
# flooded TAZ and retain the first one within the access target.  A point for
# which this is impossible is retained and explicitly flagged downstream.
road_sampling_segments = roads[["geometry"]].copy()
if road_sampling_segments.crs != demand_origins.crs:
    road_sampling_segments = road_sampling_segments.to_crs(
        demand_origins.crs
    )

initial_road_access = gpd.sjoin_nearest(
    demand_origins[["origin_id", "geometry"]],
    road_sampling_segments,
    how="left",
    distance_col="road_access_precheck_m",
)
initial_road_access = (
    initial_road_access
    .sort_values("road_access_precheck_m")
    .drop_duplicates("origin_id", keep="first")
)

long_access_ids = set(
    initial_road_access.loc[
        initial_road_access["road_access_precheck_m"]
        > DEMAND_POINT_ROAD_ACCESS_TARGET_M,
        "origin_id",
    ]
)

flooded_polygon_by_taz = (
    demand_polygons
    .set_index("TAZV41")
    .geometry
    .to_dict()
)

candidate_records = []
for _, origin_row in demand_origins[
    demand_origins["origin_id"].isin(long_access_ids)
].iterrows():
    parent_polygon = flooded_polygon_by_taz[int(origin_row["TAZV41"])]

    for candidate_order in range(DEMAND_POINT_ROAD_RESAMPLE_ATTEMPTS):
        candidate_records.append({
            "origin_id": origin_row["origin_id"],
            "candidate_order": candidate_order,
            "geometry": random_point_inside(parent_polygon, rng),
        })

resampled_origin_ids = set()

if candidate_records:
    candidate_points = gpd.GeoDataFrame(
        candidate_records,
        geometry="geometry",
        crs=demand_origins.crs,
    )
    candidate_road_access = gpd.sjoin_nearest(
        candidate_points,
        road_sampling_segments,
        how="left",
        distance_col="road_access_precheck_m",
    )
    candidate_road_access = (
        candidate_road_access
        .sort_values(["origin_id", "candidate_order", "road_access_precheck_m"])
        .drop_duplicates(
            ["origin_id", "candidate_order"],
            keep="first",
        )
    )
    acceptable_candidates = candidate_road_access[
        candidate_road_access["road_access_precheck_m"]
        <= DEMAND_POINT_ROAD_ACCESS_TARGET_M
    ]
    selected_candidates = (
        acceptable_candidates
        .sort_values(["origin_id", "candidate_order"])
        .drop_duplicates("origin_id", keep="first")
        .set_index("origin_id")
    )

    for origin_index, origin_row in demand_origins.iterrows():
        origin_id = origin_row["origin_id"]
        if origin_id in selected_candidates.index:
            demand_origins.at[origin_index, "geometry"] = (
                selected_candidates.at[origin_id, "geometry"]
            )
            resampled_origin_ids.add(origin_id)

final_road_access = gpd.sjoin_nearest(
    demand_origins[["origin_id", "geometry"]],
    road_sampling_segments,
    how="left",
    distance_col="road_access_precheck_m",
)
final_road_access = (
    final_road_access
    .sort_values("road_access_precheck_m")
    .drop_duplicates("origin_id", keep="first")
    .set_index("origin_id")["road_access_precheck_m"]
)

demand_origins["road_access_precheck_m"] = (
    demand_origins["origin_id"].map(final_road_access)
)
demand_origins["road_access_sampling_status"] = np.where(
    demand_origins["road_access_precheck_m"]
    <= DEMAND_POINT_ROAD_ACCESS_TARGET_M,
    "within_road_access_target",
    "long_off_network_connector",
)

flooded_demand_summary = pd.DataFrame([
    ("flooded_taz_with_people", len(demand_polygons)),
    ("spatial_demand_points", len(demand_origins)),
    ("people_at_alert_in_intersecting_taz", round(float(
        taz_demand.loc[
            taz_demand["flooded_area_m2"] > 0,
            "people_at_alert",
        ].sum()
    ), 2)),
    ("estimated_people_inside_flooded_area", round(
        float(demand_origins["flooded_people"].sum()),
        2,
    )),
    ("people_per_demand_point_target", DEMAND_PEOPLE_PER_POINT),
    ("maximum_points_per_taz", MAX_DEMAND_POINTS_PER_TAZ),
    ("road_access_sampling_target_m", DEMAND_POINT_ROAD_ACCESS_TARGET_M),
    ("demand_points_resampled_near_roads", len(resampled_origin_ids)),
    (
        "demand_points_still_over_road_access_target",
        int((
            demand_origins["road_access_sampling_status"]
            == "long_off_network_connector"
        ).sum()),
    ),
], columns=["metric", "value"])

display(flooded_demand_summary)

display_columns = [
    "origin_id",
    "TAZV41",
    "demand_point_index",
    "demand_point_count",
    "flooded_people",
    "parent_flooded_people",
    "road_access_precheck_m",
    "road_access_sampling_status",
    "geometry",
]

display(demand_origins[display_columns].head(15))


In [ ]:
import networkx as nx
from itertools import pairwise
from shapely.geometry import Point, LineString, MultiLineString
from shapely.ops import linemerge, substring

# ==========================================================
# Approximate routable road-network setup
# ==========================================================
# The supplied layers do not contain explicit from/to-node fields.
# Road endpoints are therefore snapped to nearby network nodes.
NETWORK_ENDPOINT_SNAP_TOLERANCE_M = 25.0

# A point is joined to its nearest road section before routing.  The model
# retains every demand point, but flags long off-network access connectors.
MAX_NETWORK_ACCESS_DISTANCE_M = 5_000.0
NETWORK_ACCESS_REVIEW_THRESHOLD_M = 500.0

road_network_roads = roads.copy()
road_network_nodes = nodes[["id", "geometry"]].copy()

if road_network_roads.crs != f"EPSG:{EPSG_ITM}":
    road_network_roads = road_network_roads.to_crs(EPSG_ITM)
if road_network_nodes.crs != f"EPSG:{EPSG_ITM}":
    road_network_nodes = road_network_nodes.to_crs(EPSG_ITM)

road_network_nodes = road_network_nodes.rename(
    columns={"id": "network_node_id"}
)
road_network_nodes["network_node_id"] = pd.to_numeric(
    road_network_nodes["network_node_id"],
    errors="coerce",
).astype("Int64")
road_network_nodes = road_network_nodes.dropna(
    subset=["network_node_id", "geometry"]
).copy()
road_network_nodes["network_node_id"] = (
    road_network_nodes["network_node_id"].astype("int64")
)

endpoint_records = []
for road_index, road_geometry in road_network_roads.geometry.items():
    if (
        road_geometry is None
        or road_geometry.is_empty
        or road_geometry.geom_type != "LineString"
    ):
        continue

    endpoint_records.extend([
        {
            "road_index": road_index,
            "endpoint": "from",
            "geometry": Point(road_geometry.coords[0]),
        },
        {
            "road_index": road_index,
            "endpoint": "to",
            "geometry": Point(road_geometry.coords[-1]),
        },
    ])

road_endpoints = gpd.GeoDataFrame(
    endpoint_records,
    geometry="geometry",
    crs=road_network_roads.crs,
)

endpoint_snaps = gpd.sjoin_nearest(
    road_endpoints,
    road_network_nodes[["network_node_id", "geometry"]],
    how="left",
    distance_col="endpoint_snap_m",
)
endpoint_snaps = (
    endpoint_snaps
    .sort_values("endpoint_snap_m")
    .drop_duplicates(["road_index", "endpoint"], keep="first")
)

endpoint_node_table = endpoint_snaps.pivot(
    index="road_index",
    columns="endpoint",
    values="network_node_id",
).reindex(columns=["from", "to"])

endpoint_distance_table = endpoint_snaps.pivot(
    index="road_index",
    columns="endpoint",
    values="endpoint_snap_m",
).reindex(columns=["from", "to"])

road_network_edges = road_network_roads[["geometry"]].copy()
road_network_edges["from_node_id"] = (
    road_network_edges.index.to_series().map(endpoint_node_table["from"])
)
road_network_edges["to_node_id"] = (
    road_network_edges.index.to_series().map(endpoint_node_table["to"])
)
road_network_edges["from_snap_m"] = (
    road_network_edges.index.to_series().map(endpoint_distance_table["from"])
)
road_network_edges["to_snap_m"] = (
    road_network_edges.index.to_series().map(endpoint_distance_table["to"])
)
road_network_edges["max_endpoint_snap_m"] = road_network_edges[
    ["from_snap_m", "to_snap_m"]
].max(axis=1)

valid_endpoint_edges = road_network_edges[
    road_network_edges["from_node_id"].notna()
    & road_network_edges["to_node_id"].notna()
    & (
        road_network_edges["max_endpoint_snap_m"]
        <= NETWORK_ENDPOINT_SNAP_TOLERANCE_M
    )
].copy()

valid_endpoint_edges["from_node_id"] = (
    valid_endpoint_edges["from_node_id"].astype("int64")
)
valid_endpoint_edges["to_node_id"] = (
    valid_endpoint_edges["to_node_id"].astype("int64")
)

self_loop_edge_count = int((
    valid_endpoint_edges["from_node_id"]
    == valid_endpoint_edges["to_node_id"]
).sum())

valid_network_edges = valid_endpoint_edges[
    valid_endpoint_edges["from_node_id"]
    != valid_endpoint_edges["to_node_id"]
].copy()

# MultiGraph keeps parallel road sections. The data has no one-way attribute,
# so this is intentionally an undirected approximation.
road_graph = nx.MultiGraph()

for road_index, edge in valid_network_edges.iterrows():
    road_graph.add_edge(
        int(edge["from_node_id"]),
        int(edge["to_node_id"]),
        length_m=float(edge.geometry.length),
        geometry=edge.geometry,
        road_index=int(road_index),
    )

if road_graph.number_of_edges() == 0:
    raise RuntimeError("No routable road edges were created.")

largest_component_nodes = max(
    nx.connected_components(road_graph),
    key=len,
)
road_graph = road_graph.subgraph(largest_component_nodes).copy()

# These are the road sections retained in the routable component.  Origin
# and refuge points are snapped to these *sections*, rather than to junctions.
network_edges_gdf = valid_network_edges[
    valid_network_edges["from_node_id"].isin(road_graph.nodes)
    & valid_network_edges["to_node_id"].isin(road_graph.nodes)
].copy().reset_index(drop=True)
network_edges_gdf["network_edge_id"] = np.arange(
    len(network_edges_gdf),
    dtype="int64",
)
network_edges_gdf["edge_length_m"] = network_edges_gdf.geometry.length

if len(network_edges_gdf) == 0:
    raise RuntimeError("No road sections remain in the main network component.")

network_edge_geometry_by_id = (
    network_edges_gdf
    .set_index("network_edge_id")
    .geometry
    .to_dict()
)

network_nodes_gdf = road_network_nodes[
    road_network_nodes["network_node_id"].isin(road_graph.nodes)
].copy()

network_node_geometry_by_id = (
    network_nodes_gdf
    .set_index("network_node_id")
    .geometry
    .to_dict()
)

network_hazard_union = hazard_area.union_all()
network_nodes_gdf["outside_flood_area"] = ~network_nodes_gdf.geometry.within(
    network_hazard_union
)

network_exit_node_ids = network_nodes_gdf.loc[
    network_nodes_gdf["outside_flood_area"],
    "network_node_id",
].tolist()

if len(network_exit_node_ids) == 0:
    raise RuntimeError("No network nodes were found outside the flood area.")

# One multi-source search gives shortest road distance from every graph node
# to its closest network node outside the flood area.
network_exit_distance_m, network_exit_paths = nx.multi_source_dijkstra(
    road_graph,
    sources=network_exit_node_ids,
    weight="length_m",
)
network_exit_node_by_node_id = {
    int(node_id): int(path[0])
    for node_id, path in network_exit_paths.items()
}


def snap_points_to_road_network(points_gdf, id_column):
    """Snap points to nearest road sections and retain along-road position."""
    points_to_snap = points_gdf[[id_column, "geometry"]].copy()

    if points_to_snap.crs != network_edges_gdf.crs:
        points_to_snap = points_to_snap.to_crs(network_edges_gdf.crs)

    snapped = gpd.sjoin_nearest(
        points_to_snap,
        network_edges_gdf[
            [
                "network_edge_id",
                "from_node_id",
                "to_node_id",
                "edge_length_m",
                "geometry",
            ]
        ],
        how="left",
        distance_col="network_access_distance_m",
    )

    snapped = (
        snapped
        .sort_values("network_access_distance_m")
        .drop_duplicates(id_column, keep="first")
        .drop(columns=["index_right"], errors="ignore")
    )

    for column in [
        "network_edge_id",
        "from_node_id",
        "to_node_id",
    ]:
        snapped[column] = pd.to_numeric(
            snapped[column],
            errors="coerce",
        ).astype("Int64")

    snapped["edge_length_m"] = pd.to_numeric(
        snapped["edge_length_m"],
        errors="coerce",
    )
    snapped["edge_projection_m"] = np.nan
    snapped["snapped_road_point"] = None

    valid_snaps = snapped[
        snapped["network_edge_id"].notna()
        & snapped["edge_length_m"].notna()
    ]

    for snap_index, snap_row in valid_snaps.iterrows():
        edge_id = int(snap_row["network_edge_id"])
        edge_geometry = network_edge_geometry_by_id[edge_id]
        edge_length_m = float(snap_row["edge_length_m"])
        edge_projection_m = min(
            max(float(edge_geometry.project(snap_row.geometry)), 0.0),
            edge_length_m,
        )

        snapped.at[snap_index, "edge_projection_m"] = edge_projection_m
        snapped.at[snap_index, "snapped_road_point"] = (
            edge_geometry.interpolate(edge_projection_m)
        )

    return gpd.GeoDataFrame(
        snapped,
        geometry="geometry",
        crs=points_to_snap.crs,
    )


def endpoint_options_from_snap(snap_row):
    """Return both graph endpoints reachable along a snapped road section."""
    required_columns = [
        "network_edge_id",
        "from_node_id",
        "to_node_id",
        "edge_length_m",
        "edge_projection_m",
    ]

    if any(pd.isna(snap_row[column]) for column in required_columns):
        return []

    edge_length_m = float(snap_row["edge_length_m"])
    edge_projection_m = min(
        max(float(snap_row["edge_projection_m"]), 0.0),
        edge_length_m,
    )

    return [
        {
            "node_id": int(snap_row["from_node_id"]),
            "edge_distance_m": edge_projection_m,
            "edge_endpoint_projection_m": 0.0,
        },
        {
            "node_id": int(snap_row["to_node_id"]),
            "edge_distance_m": edge_length_m - edge_projection_m,
            "edge_endpoint_projection_m": edge_length_m,
        },
    ]


def best_road_edge(from_node, to_node):
    """Return the shortest parallel road section between two graph nodes."""
    return min(
        road_graph[from_node][to_node].values(),
        key=lambda edge: edge["length_m"],
    )


def _append_nonzero_line(line_parts, geometry):
    """Append non-empty line components without creating zero-length parts."""
    if geometry is None or geometry.is_empty:
        return

    if geometry.geom_type == "LineString":
        if geometry.length > 0.01:
            line_parts.append(geometry)
    elif geometry.geom_type == "MultiLineString":
        for part in geometry.geoms:
            _append_nonzero_line(line_parts, part)


def _road_substring(edge_id, start_m, end_m):
    """Return the traversed part of a road section, in travel direction."""
    if abs(float(end_m) - float(start_m)) <= 0.01:
        return None

    return substring(
        network_edge_geometry_by_id[int(edge_id)],
        float(start_m),
        float(end_m),
    )


def route_geometry_from_spec(route_spec):
    """Build route geometry from access connectors and actual road sections."""
    line_parts = []

    origin_point = route_spec["origin_point"]
    origin_road_point = route_spec["origin_snapped_road_point"]
    _append_nonzero_line(
        line_parts,
        LineString([origin_point, origin_road_point]),
    )

    if route_spec.get("direct_same_edge", False):
        _append_nonzero_line(
            line_parts,
            _road_substring(
                route_spec["origin_edge_id"],
                route_spec["origin_projection_m"],
                route_spec["destination_projection_m"],
            ),
        )
    else:
        _append_nonzero_line(
            line_parts,
            _road_substring(
                route_spec["origin_edge_id"],
                route_spec["origin_projection_m"],
                route_spec["origin_endpoint_projection_m"],
            ),
        )

        for from_node, to_node in pairwise(route_spec["node_path"]):
            _append_nonzero_line(
                line_parts,
                best_road_edge(from_node, to_node)["geometry"],
            )

        if route_spec["route_kind"] == "building":
            _append_nonzero_line(
                line_parts,
                _road_substring(
                    route_spec["destination_edge_id"],
                    route_spec["destination_endpoint_projection_m"],
                    route_spec["destination_projection_m"],
                ),
            )

    if route_spec["route_kind"] == "building":
        _append_nonzero_line(
            line_parts,
            LineString([
                route_spec["destination_snapped_road_point"],
                route_spec["destination_point"],
            ]),
        )

    if not line_parts:
        raise RuntimeError(
            "No non-zero geometry was created for a road-network route."
        )

    if len(line_parts) == 1:
        return line_parts[0]

    # A MultiLineString intentionally preserves small geometry offsets between
    # source road endpoints and the separately supplied junction layer.
    return linemerge(
        MultiLineString([list(line.coords) for line in line_parts])
    )


network_routing_summary = pd.DataFrame([
    (
        "routing_method",
        "approximate undirected shortest paths on supplied road sections",
    ),
    (
        "point_snap_method",
        "nearest routable road section, then travel along that section",
    ),
    ("road_sections_total", len(road_network_roads)),
    ("road_sections_with_both_endpoints_snapped", len(valid_endpoint_edges)),
    ("self_loop_sections_excluded", self_loop_edge_count),
    ("road_edges_in_main_component", road_graph.number_of_edges()),
    ("road_nodes_in_main_component", road_graph.number_of_nodes()),
    ("road_endpoint_snap_tolerance_m", NETWORK_ENDPOINT_SNAP_TOLERANCE_M),
    ("maximum_origin_or_destination_access_m", MAX_NETWORK_ACCESS_DISTANCE_M),
    ("access_distance_review_threshold_m", NETWORK_ACCESS_REVIEW_THRESHOLD_M),
    ("safe_exit_network_nodes", len(network_exit_node_ids)),
], columns=["metric", "value"])

display(network_routing_summary)


In [ ]:
# ==========================================================
# Capacity-constrained assignment using shortest road-network paths
# ==========================================================
# Route distance = point-to-road access connector + road-section travel
#                  + shortest path on the routable road graph.

required_network_variables = [
    "road_graph",
    "network_nodes_gdf",
    "network_edge_geometry_by_id",
    "network_exit_distance_m",
    "network_exit_paths",
    "network_node_geometry_by_id",
    "snap_points_to_road_network",
    "endpoint_options_from_snap",
]

missing_network_variables = [
    variable for variable in required_network_variables
    if variable not in globals()
]

if missing_network_variables:
    raise RuntimeError(
        "Run the preceding road-network setup cell first. Missing: "
        + ", ".join(missing_network_variables)
    )

if "vertical_refuge_suitability" in globals():
    refuge_destinations = vertical_refuge_suitability[
        vertical_refuge_suitability["refuge_assignment_eligible"]
    ].copy()
    assignment_capacity_field = "planned_effective_capacity_people"
    assignment_destination_note = "planned eligible refuges only"
else:
    refuge_destinations = vertical_refuge_capacity.copy()
    assignment_capacity_field = "capacity_people"
    assignment_destination_note = "all vertical-refuge capacity buildings"

if refuge_destinations.crs != f"EPSG:{EPSG_ITM}":
    refuge_destinations = refuge_destinations.to_crs(EPSG_ITM)

refuge_destinations[assignment_capacity_field] = pd.to_numeric(
    refuge_destinations[assignment_capacity_field],
    errors="coerce",
).fillna(0.0)

refuge_destinations = refuge_destinations[
    refuge_destinations[assignment_capacity_field] > 0
].copy()

refuge_destinations["assignment_capacity_people"] = (
    refuge_destinations[assignment_capacity_field].astype(float)
)
refuge_destinations["destination_id"] = (
    "B_" + refuge_destinations["id_binyan"].astype(str)
)
refuge_destinations["destination_type"] = "vertical_refuge_building"

if refuge_destinations["destination_id"].duplicated().any():
    raise ValueError("Duplicate refuge destination IDs were found.")

for column in [
    "capacity_people",
    "capacity_class",
    "refuge_suitability_class",
    "access_probability",
]:
    if column not in refuge_destinations.columns:
        refuge_destinations[column] = np.nan

refuge_destination_points = refuge_destinations.copy()
refuge_destination_points["geometry"] = (
    refuge_destination_points.geometry.representative_point()
)
refuge_destination_points = gpd.GeoDataFrame(
    refuge_destination_points,
    geometry="geometry",
    crs=refuge_destinations.crs,
)

# Snap origins and buildings to road *sections*.  Each point can then reach
# either endpoint of its section, allowing a shortest-path calculation that
# does not force it first to the nearest junction.
origin_network_snaps = snap_points_to_road_network(
    demand_origins,
    "origin_id",
)
destination_network_snaps = snap_points_to_road_network(
    refuge_destination_points,
    "destination_id",
)

origin_snap_missing_columns = [
    "network_edge_id",
    "from_node_id",
    "to_node_id",
    "edge_projection_m",
    "snapped_road_point",
]
invalid_origins = origin_network_snaps[
    origin_network_snaps[origin_snap_missing_columns].isna().any(axis=1)
    | (
        origin_network_snaps["network_access_distance_m"]
        > MAX_NETWORK_ACCESS_DISTANCE_M
    )
].copy()

if len(invalid_origins) > 0:
    raise RuntimeError(
        f"{len(invalid_origins)} demand origins could not be connected to the "
        f"road network within {MAX_NETWORK_ACCESS_DISTANCE_M:.0f} m."
    )

invalid_destinations = destination_network_snaps[
    destination_network_snaps[origin_snap_missing_columns].isna().any(axis=1)
    | (
        destination_network_snaps["network_access_distance_m"]
        > MAX_NETWORK_ACCESS_DISTANCE_M
    )
].copy()

routable_destination_snaps = destination_network_snaps.drop(
    invalid_destinations.index
).copy()
excluded_destination_count = len(invalid_destinations)

if len(routable_destination_snaps) == 0:
    raise RuntimeError(
        "No refuge destinations could be connected to the road network."
    )

origin_long_access_ids = set(
    origin_network_snaps.loc[
        origin_network_snaps["network_access_distance_m"]
        > NETWORK_ACCESS_REVIEW_THRESHOLD_M,
        "origin_id",
    ]
)
destination_long_access_count = int((
    routable_destination_snaps["network_access_distance_m"]
    > NETWORK_ACCESS_REVIEW_THRESHOLD_M
).sum())

origin_long_access_people = float(
    demand_origins.loc[
        demand_origins["origin_id"].isin(origin_long_access_ids),
        "flooded_people",
    ].sum()
)

routable_destination_ids = set(
    routable_destination_snaps["destination_id"]
)
refuge_destinations = refuge_destinations[
    refuge_destinations["destination_id"].isin(routable_destination_ids)
].copy()
refuge_destination_points = refuge_destination_points[
    refuge_destination_points["destination_id"].isin(routable_destination_ids)
].copy()

destination_by_id = refuge_destination_points.set_index(
    "destination_id",
    drop=False,
)
origin_snap_by_id = origin_network_snaps.set_index(
    "origin_id",
    drop=False,
)
destination_snap_by_id = routable_destination_snaps.set_index(
    "destination_id",
    drop=False,
)


# In an undirected graph, origin-to-refuge and refuge-to-origin distances are
# identical.  Precompute once per unique refuge endpoint instead of running a
# full Dijkstra search for every demand point.
destination_dijkstra_by_node = {}

for _, destination_snap in routable_destination_snaps.iterrows():
    for destination_option in endpoint_options_from_snap(destination_snap):
        destination_node_id = destination_option["node_id"]

        if destination_node_id not in destination_dijkstra_by_node:
            destination_dijkstra_by_node[destination_node_id] = (
                nx.single_source_dijkstra(
                    road_graph,
                    source=destination_node_id,
                    weight="length_m",
                )
            )


def direct_same_edge_route(origin_snap, destination_snap, destination_id):
    """Return direct travel on a shared road section, when available."""
    if int(origin_snap["network_edge_id"]) != int(
        destination_snap["network_edge_id"]
    ):
        return None

    road_distance_m = abs(
        float(origin_snap["edge_projection_m"])
        - float(destination_snap["edge_projection_m"])
    )

    return {
        "destination_id": destination_id,
        "origin_access_m": float(
            origin_snap["network_access_distance_m"]
        ),
        "destination_access_m": float(
            destination_snap["network_access_distance_m"]
        ),
        "origin_edge_id": int(origin_snap["network_edge_id"]),
        "destination_edge_id": int(destination_snap["network_edge_id"]),
        "origin_projection_m": float(origin_snap["edge_projection_m"]),
        "destination_projection_m": float(
            destination_snap["edge_projection_m"]
        ),
        "origin_snapped_road_point": origin_snap["snapped_road_point"],
        "destination_snapped_road_point": destination_snap[
            "snapped_road_point"
        ],
        "origin_road_segment_m": 0.0,
        "destination_road_segment_m": 0.0,
        "network_distance_m": road_distance_m,
        "route_distance_m": (
            float(origin_snap["network_access_distance_m"])
            + road_distance_m
            + float(destination_snap["network_access_distance_m"])
        ),
        "origin_network_node_id": np.nan,
        "destination_network_node_id": np.nan,
        "origin_endpoint_projection_m": np.nan,
        "destination_endpoint_projection_m": np.nan,
        "node_path": [],
        "direct_same_edge": True,
    }


def shortest_building_route(
    origin_snap,
    destination_snap,
    destination_id,
    destination_dijkstra_by_node,
):
    """Choose the shortest viable road path from one origin to one refuge."""
    route_candidates = []

    direct_route = direct_same_edge_route(
        origin_snap,
        destination_snap,
        destination_id,
    )
    if direct_route is not None:
        route_candidates.append(direct_route)

    origin_options = endpoint_options_from_snap(origin_snap)
    destination_options = endpoint_options_from_snap(destination_snap)

    for destination_option in destination_options:
        destination_node_id = destination_option["node_id"]
        network_distances, network_paths = (
            destination_dijkstra_by_node[destination_node_id]
        )

        for origin_option in origin_options:
            origin_node_id = origin_option["node_id"]

            if origin_node_id not in network_distances:
                continue

            network_distance_m = float(network_distances[origin_node_id])
            route_distance_m = (
                float(origin_snap["network_access_distance_m"])
                + float(origin_option["edge_distance_m"])
                + network_distance_m
                + float(destination_option["edge_distance_m"])
                + float(destination_snap["network_access_distance_m"])
            )

            # The cached path is destination -> origin, while the stored route
            # geometry and assignment record use origin -> destination.
            node_path = list(reversed(network_paths[origin_node_id]))

            route_candidates.append({
                "destination_id": destination_id,
                "origin_access_m": float(
                    origin_snap["network_access_distance_m"]
                ),
                "destination_access_m": float(
                    destination_snap["network_access_distance_m"]
                ),
                "origin_edge_id": int(origin_snap["network_edge_id"]),
                "destination_edge_id": int(
                    destination_snap["network_edge_id"]
                ),
                "origin_projection_m": float(
                    origin_snap["edge_projection_m"]
                ),
                "destination_projection_m": float(
                    destination_snap["edge_projection_m"]
                ),
                "origin_snapped_road_point": origin_snap[
                    "snapped_road_point"
                ],
                "destination_snapped_road_point": destination_snap[
                    "snapped_road_point"
                ],
                "origin_road_segment_m": float(
                    origin_option["edge_distance_m"]
                ),
                "destination_road_segment_m": float(
                    destination_option["edge_distance_m"]
                ),
                "network_distance_m": network_distance_m,
                "route_distance_m": route_distance_m,
                "origin_network_node_id": origin_node_id,
                "destination_network_node_id": destination_node_id,
                "origin_endpoint_projection_m": float(
                    origin_option["edge_endpoint_projection_m"]
                ),
                "destination_endpoint_projection_m": float(
                    destination_option["edge_endpoint_projection_m"]
                ),
                "node_path": node_path,
                "direct_same_edge": False,
            })

    if not route_candidates:
        return None

    return min(
        route_candidates,
        key=lambda route: route["route_distance_m"],
    )

def shortest_outside_route(origin_snap):
    """Choose the nearest graph node outside the flood area for one origin."""
    route_candidates = []

    for origin_option in endpoint_options_from_snap(origin_snap):
        origin_node_id = origin_option["node_id"]

        if origin_node_id not in network_exit_distance_m:
            continue

        exit_network_distance_m = float(
            network_exit_distance_m[origin_node_id]
        )
        # multi_source_dijkstra stores paths from the source exit to the node.
        node_path = list(reversed(network_exit_paths[origin_node_id]))
        exit_node_id = int(node_path[-1])

        route_candidates.append({
            "origin_access_m": float(
                origin_snap["network_access_distance_m"]
            ),
            "origin_edge_id": int(origin_snap["network_edge_id"]),
            "origin_projection_m": float(origin_snap["edge_projection_m"]),
            "origin_snapped_road_point": origin_snap[
                "snapped_road_point"
            ],
            "origin_road_segment_m": float(
                origin_option["edge_distance_m"]
            ),
            "network_distance_m": exit_network_distance_m,
            "route_distance_m": (
                float(origin_snap["network_access_distance_m"])
                + float(origin_option["edge_distance_m"])
                + exit_network_distance_m
            ),
            "origin_network_node_id": origin_node_id,
            "origin_endpoint_projection_m": float(
                origin_option["edge_endpoint_projection_m"]
            ),
            "exit_node_id": exit_node_id,
            "exit_point": network_node_geometry_by_id[exit_node_id],
            "node_path": node_path,
        })

    if not route_candidates:
        return None

    return min(
        route_candidates,
        key=lambda route: route["route_distance_m"],
    )


# Pre-calculate shortest routes from every demand point.
origin_info = []

for origin_index, origin in demand_origins.iterrows():
    origin_id = origin["origin_id"]
    origin_snap = origin_snap_by_id.loc[origin_id]

    outside_route = shortest_outside_route(origin_snap)
    if outside_route is None:
        raise RuntimeError(
            f"No outside-flood network route was found for {origin_id}."
        )

    building_routes = []

    for destination_id, destination in destination_by_id.iterrows():
        destination_snap = destination_snap_by_id.loc[destination_id]
        route = shortest_building_route(
            origin_snap,
            destination_snap,
            destination_id,
            destination_dijkstra_by_node,
        )

        if route is not None:
            building_routes.append(route)

    building_routes.sort(key=lambda route: route["route_distance_m"])

    nearest_building_route_m = (
        building_routes[0]["route_distance_m"]
        if building_routes
        else np.inf
    )

    origin_info.append({
        "origin_index": origin_index,
        "origin_id": origin_id,
        "origin_snap": origin_snap,
        "outside_route": outside_route,
        "nearest_building_route_m": nearest_building_route_m,
        "building_routes": building_routes,
        "refuge_priority": (
            outside_route["route_distance_m"]
            - nearest_building_route_m
        ),
        "demand_people": float(origin["flooded_people"]),
    })

origin_info = sorted(
    origin_info,
    key=lambda item: (
        item["refuge_priority"],
        item["outside_route"]["route_distance_m"],
    ),
    reverse=True,
)

remaining_capacity = (
    refuge_destinations.set_index("destination_id")[
        "assignment_capacity_people"
    ].to_dict()
)

allocation_records = []
assignment_route_specs = {}
EPSILON = 1e-6

for info in origin_info:
    origin_index = info["origin_index"]
    origin = demand_origins.loc[origin_index]
    original_demand = info["demand_people"]
    remaining_people = original_demand
    assignment_rank = 0

    # A refuge is used only if its complete road-route distance is shorter
    # than taking the road network to the nearest node outside the flood area.
    for route in info["building_routes"]:
        if route["route_distance_m"] >= info["outside_route"]["route_distance_m"]:
            break

        destination_id = route["destination_id"]
        available_capacity = remaining_capacity[destination_id]

        if available_capacity <= EPSILON:
            continue

        assigned_people = min(remaining_people, available_capacity)
        destination = destination_by_id.loc[destination_id]
        assignment_id = f"A_{origin['origin_id']}_{assignment_rank:02d}"

        record = origin.drop(labels="geometry").to_dict()
        record.update({
            "geometry": origin.geometry,
            "assignment_id": assignment_id,
            "origin_flooded_people": original_demand,
            "flooded_people": assigned_people,
            "chosen_destination_type": "vertical_refuge_building",
            "chosen_destination_id": destination_id,
            "chosen_distance_m": route["route_distance_m"],
            "distance_to_building_m": info["nearest_building_route_m"],
            "distance_to_outside_m": info["outside_route"]["route_distance_m"],
            "network_path_distance_m": route["network_distance_m"],
            "origin_network_access_m": route["origin_access_m"],
            "destination_network_access_m": route[
                "destination_access_m"
            ],
            "origin_road_segment_m": route["origin_road_segment_m"],
            "destination_road_segment_m": route[
                "destination_road_segment_m"
            ],
            "origin_network_node_id": route["origin_network_node_id"],
            "destination_network_node_id": route[
                "destination_network_node_id"
            ],
            "origin_road_edge_id": route["origin_edge_id"],
            "destination_road_edge_id": route["destination_edge_id"],
            "assigned_to_building_people": assigned_people,
            "assigned_to_outside_people": 0.0,
            "assignment_rank": assignment_rank,
            "destination_capacity_people": destination[
                "assignment_capacity_people"
            ],
            "remaining_capacity_after_assignment": (
                available_capacity - assigned_people
            ),
            "refuge_suitability_class": destination[
                "refuge_suitability_class"
            ],
            "access_probability": destination["access_probability"],
            "_origin_index": origin_index,
        })
        allocation_records.append(record)

        assignment_route_specs[assignment_id] = {
            "route_kind": "building",
            "origin_point": origin.geometry,
            "origin_snapped_road_point": route[
                "origin_snapped_road_point"
            ],
            "origin_edge_id": route["origin_edge_id"],
            "origin_projection_m": route["origin_projection_m"],
            "origin_endpoint_projection_m": route[
                "origin_endpoint_projection_m"
            ],
            "node_path": route["node_path"],
            "destination_point": destination.geometry,
            "destination_snapped_road_point": route[
                "destination_snapped_road_point"
            ],
            "destination_edge_id": route["destination_edge_id"],
            "destination_projection_m": route[
                "destination_projection_m"
            ],
            "destination_endpoint_projection_m": route[
                "destination_endpoint_projection_m"
            ],
            "direct_same_edge": route["direct_same_edge"],
        }

        remaining_capacity[destination_id] -= assigned_people
        remaining_people -= assigned_people
        assignment_rank += 1

        if remaining_people <= EPSILON:
            break

    # Demand not sent to a closer refuge follows the road network to the
    # closest node that lies outside the flood area.
    if remaining_people > EPSILON:
        outside_route = info["outside_route"]
        assignment_id = f"A_{origin['origin_id']}_{assignment_rank:02d}"

        record = origin.drop(labels="geometry").to_dict()
        record.update({
            "geometry": origin.geometry,
            "assignment_id": assignment_id,
            "origin_flooded_people": original_demand,
            "flooded_people": remaining_people,
            "chosen_destination_type": "outside_flood_area",
            "chosen_destination_id": f"OUTSIDE_{origin['origin_id']}",
            "chosen_distance_m": outside_route["route_distance_m"],
            "distance_to_building_m": info["nearest_building_route_m"],
            "distance_to_outside_m": outside_route["route_distance_m"],
            "network_path_distance_m": outside_route["network_distance_m"],
            "origin_network_access_m": outside_route["origin_access_m"],
            "destination_network_access_m": 0.0,
            "origin_road_segment_m": outside_route[
                "origin_road_segment_m"
            ],
            "destination_road_segment_m": 0.0,
            "origin_network_node_id": outside_route[
                "origin_network_node_id"
            ],
            "destination_network_node_id": outside_route["exit_node_id"],
            "origin_road_edge_id": outside_route["origin_edge_id"],
            "destination_road_edge_id": np.nan,
            "assigned_to_building_people": 0.0,
            "assigned_to_outside_people": remaining_people,
            "assignment_rank": assignment_rank,
            "destination_capacity_people": np.nan,
            "remaining_capacity_after_assignment": np.nan,
            "refuge_suitability_class": None,
            "access_probability": np.nan,
            "_origin_index": origin_index,
        })
        allocation_records.append(record)

        assignment_route_specs[assignment_id] = {
            "route_kind": "outside",
            "origin_point": origin.geometry,
            "origin_snapped_road_point": outside_route[
                "origin_snapped_road_point"
            ],
            "origin_edge_id": outside_route["origin_edge_id"],
            "origin_projection_m": outside_route["origin_projection_m"],
            "origin_endpoint_projection_m": outside_route[
                "origin_endpoint_projection_m"
            ],
            "node_path": outside_route["node_path"],
            "destination_point": outside_route["exit_point"],
            "direct_same_edge": False,
        }

assignments = gpd.GeoDataFrame(
    allocation_records,
    geometry="geometry",
    crs=demand_origins.crs,
)

outside_exit_points = assignments[
    assignments["chosen_destination_type"] == "outside_flood_area"
].copy()

if len(outside_exit_points) > 0:
    outside_exit_points["geometry"] = outside_exit_points[
        "destination_network_node_id"
    ].map(network_node_geometry_by_id)
    outside_exit_points = gpd.GeoDataFrame(
        outside_exit_points,
        geometry="geometry",
        crs=assignments.crs,
    )

assignments = assignments.drop(columns="_origin_index")
outside_exit_points = outside_exit_points.drop(columns="_origin_index")

# Validation: preserve all demand and never exceed a refuge capacity.
input_demand = float(demand_origins["flooded_people"].sum())
assigned_demand = float(assignments["flooded_people"].sum())

if not np.isclose(input_demand, assigned_demand, atol=EPSILON):
    raise RuntimeError(
        f"Demand conservation failed: input={input_demand}, assigned={assigned_demand}"
    )

building_assignments = assignments[
    assignments["chosen_destination_type"] == "vertical_refuge_building"
]
assigned_by_building = building_assignments.groupby(
    "chosen_destination_id"
)["flooded_people"].sum()

capacity_by_building = refuge_destinations.set_index(
    "destination_id"
)["assignment_capacity_people"]

overloaded_buildings = assigned_by_building[
    assigned_by_building
    > capacity_by_building.reindex(assigned_by_building.index) + EPSILON
]

if len(overloaded_buildings) > 0:
    raise RuntimeError("Capacity constraint failed for one or more buildings.")

assignment_summary = pd.DataFrame([
    (
        "assignment_method",
        "capacity-constrained shortest road-network path",
    ),
    (
        "distance_components",
        "access connector + road section + graph path + road section + access",
    ),
    ("network_type", "undirected approximate road topology"),
    ("assignment_destination_note", assignment_destination_note),
    ("refuge_endpoint_dijkstra_searches", len(destination_dijkstra_by_node)),
    ("origin_points", int(demand_origins["origin_id"].nunique())),
    ("assignment_records", len(assignments)),
    (
        "origins_over_access_review_threshold",
        len(origin_long_access_ids),
    ),
    (
        "people_at_origins_over_access_review_threshold",
        round(origin_long_access_people, 2),
    ),
    (
        "destinations_over_access_review_threshold",
        destination_long_access_count,
    ),
    ("access_review_threshold_m", NETWORK_ACCESS_REVIEW_THRESHOLD_M),
    ("people_assigned_to_buildings", round(
        float(assignments["assigned_to_building_people"].sum()), 2
    )),
    ("people_assigned_to_outside", round(
        float(assignments["assigned_to_outside_people"].sum()), 2
    )),
    ("total_people_assigned", round(assigned_demand, 2)),
    ("mean_chosen_network_distance_m", round(
        float(assignments["chosen_distance_m"].mean()), 2
    )),
    ("eligible_refuge_destinations", len(refuge_destinations)),
    ("destinations_excluded_as_unroutable", excluded_destination_count),
    ("unused_refuge_capacity", round(sum(remaining_capacity.values()), 2)),
], columns=["metric", "value"])

display(assignment_summary)
display(
    assignments[
        [
            "origin_id",
            "flooded_people",
            "chosen_destination_type",
            "chosen_destination_id",
            "chosen_distance_m",
            "network_path_distance_m",
            "origin_road_segment_m",
            "destination_road_segment_m",
            "origin_network_access_m",
            "destination_network_access_m",
        ]
    ].head(20)
)


## Deep Safe Exit Gates

This section replaces nearest-boundary exits with fixed, deep safe gates. The 4,000-person limit per gate is a planning dispersion cap; operational capacity is tested in Aimsun.


In [ ]:
# ==========================================================
# Replace nearest-boundary exits with deep safe exit gates
# ==========================================================
from shapely.ops import nearest_points

# These are planning gates, not certified road capacities. They were screened
# on the current approximate road graph to be at least 1 km outside the
# inundation polygon and at least 1 km east of the local inundation boundary.
# Aimsun remains the place to test operations.
SAFE_EXIT_GATE_MIN_HAZARD_DISTANCE_M = 1_000.0
SAFE_EXIT_GATE_MIN_EAST_OFFSET_M = 1_000.0
SAFE_EXIT_GATE_PLANNING_CAPACITY_PEOPLE = 4_000.0

SAFE_EXIT_GATE_DEFINITIONS = [
    {"gate_id": "SAFE_GATE_01", "network_node_id": 96962},
    {"gate_id": "SAFE_GATE_02", "network_node_id": 95877},
    {"gate_id": "SAFE_GATE_03", "network_node_id": 13503291},
    {"gate_id": "SAFE_GATE_04", "network_node_id": 89032},
    {"gate_id": "SAFE_GATE_05", "network_node_id": 22198},
    {"gate_id": "SAFE_GATE_06", "network_node_id": 75816},
    {"gate_id": "SAFE_GATE_07", "network_node_id": 315369},
    {"gate_id": "SAFE_GATE_08", "network_node_id": 315301},
]

if (
    "safe_exit_gate_id" in assignments.columns
    and assignments["safe_exit_gate_id"].notna().any()
):
    raise RuntimeError(
        "Safe exit gates were already applied. Run the preceding "
        "capacity-constrained assignment cell again before reapplying."
    )

safe_exit_gates = pd.DataFrame(SAFE_EXIT_GATE_DEFINITIONS)
safe_exit_gates["network_node_id"] = safe_exit_gates[
    "network_node_id"
].astype("int64")

missing_gate_nodes = sorted(
    set(safe_exit_gates["network_node_id"]) - set(road_graph.nodes)
)
if missing_gate_nodes:
    raise RuntimeError(
        "Selected safe-gate nodes are not connected to the main road graph: "
        f"{missing_gate_nodes}"
    )

safe_exit_gates["geometry"] = safe_exit_gates[
    "network_node_id"
].map(network_node_geometry_by_id)
if safe_exit_gates["geometry"].isna().any():
    raise RuntimeError("A selected safe gate has no node geometry.")

safe_exit_gates = gpd.GeoDataFrame(
    safe_exit_gates,
    geometry="geometry",
    crs=network_nodes_gdf.crs,
)
safe_exit_gates["gate_name"] = safe_exit_gates["gate_id"].str.replace(
    "_", " ", regex=False
)
safe_exit_gates["safety_distance_m"] = safe_exit_gates.geometry.distance(
    network_hazard_union
)
hazard_boundary_union = hazard_boundary.union_all()
safe_exit_gates["east_offset_m"] = safe_exit_gates.geometry.map(
    lambda point: point.x - nearest_points(point, hazard_boundary_union)[1].x
)
safe_exit_gates["network_node_degree"] = safe_exit_gates[
    "network_node_id"
].map(dict(road_graph.degree()))
safe_exit_gates["planning_capacity_people"] = (
    SAFE_EXIT_GATE_PLANNING_CAPACITY_PEOPLE
)

unsafe_gates = safe_exit_gates[
    (safe_exit_gates["safety_distance_m"] < SAFE_EXIT_GATE_MIN_HAZARD_DISTANCE_M)
    | (safe_exit_gates["east_offset_m"] < SAFE_EXIT_GATE_MIN_EAST_OFFSET_M)
    | (safe_exit_gates["network_node_degree"] < 3)
]
if len(unsafe_gates) > 0:
    raise RuntimeError(
        "One or more selected gates fail the safety or connectivity screen: "
        f"{unsafe_gates[['gate_id', 'safety_distance_m', 'east_offset_m', 'network_node_degree']].to_dict('records')}"
    )

outside_mask_before_gates = assignments[
    "chosen_destination_type"
].eq("outside_flood_area")
outside_assignments_before_gates = assignments.loc[
    outside_mask_before_gates
].copy()
if outside_assignments_before_gates.empty:
    raise RuntimeError("No outside-flood demand is available for gate assignment.")

outside_people_before_gates = float(
    outside_assignments_before_gates["flooded_people"].sum()
)
total_gate_planning_capacity = float(
    safe_exit_gates["planning_capacity_people"].sum()
)
if outside_people_before_gates > total_gate_planning_capacity + EPSILON:
    raise RuntimeError(
        "Outside demand exceeds combined safe-gate planning capacity. "
        "Add gates or raise the explicitly documented cap."
    )

# One Dijkstra search per fixed gate. The graph is still the approximate,
# undirected topology documented in the preceding network cell.
safe_gate_dijkstra_by_id = {
    gate.gate_id: nx.single_source_dijkstra(
        road_graph,
        source=int(gate.network_node_id),
        weight="length_m",
    )
    for gate in safe_exit_gates.itertuples()
}
safe_gate_by_id = safe_exit_gates.set_index("gate_id", drop=False)


def routes_to_safe_exit_gates(origin_snap):
    """Return the shortest road-network route to each reachable gate."""
    shortest_route_by_gate = {}

    for gate in safe_exit_gates.itertuples():
        network_distances, network_paths = safe_gate_dijkstra_by_id[
            gate.gate_id
        ]

        for origin_option in endpoint_options_from_snap(origin_snap):
            origin_node_id = int(origin_option["node_id"])
            if origin_node_id not in network_distances:
                continue

            network_distance_m = float(network_distances[origin_node_id])
            route = {
                "gate_id": gate.gate_id,
                "destination_point": gate.geometry,
                "destination_network_node_id": int(gate.network_node_id),
                "origin_access_m": float(origin_snap["network_access_distance_m"]),
                "origin_edge_id": int(origin_snap["network_edge_id"]),
                "origin_projection_m": float(origin_snap["edge_projection_m"]),
                "origin_snapped_road_point": origin_snap["snapped_road_point"],
                "origin_road_segment_m": float(origin_option["edge_distance_m"]),
                "origin_network_node_id": origin_node_id,
                "origin_endpoint_projection_m": float(
                    origin_option["edge_endpoint_projection_m"]
                ),
                "network_distance_m": network_distance_m,
                "route_distance_m": (
                    float(origin_snap["network_access_distance_m"])
                    + float(origin_option["edge_distance_m"])
                    + network_distance_m
                ),
                # Dijkstra stores gate -> origin; output geometry needs origin -> gate.
                "node_path": list(reversed(network_paths[origin_node_id])),
            }
            current_best = shortest_route_by_gate.get(gate.gate_id)
            if (
                current_best is None
                or route["route_distance_m"] < current_best["route_distance_m"]
            ):
                shortest_route_by_gate[gate.gate_id] = route

    return sorted(
        shortest_route_by_gate.values(),
        key=lambda route: (route["route_distance_m"], route["gate_id"]),
    )


# Allocate only the demand that remained after the closer-refuge step.
# Regret-greedy prioritizes origins whose next-best gate is much farther away.
# A source is split only if its preferred gate reaches the planning cap.
safe_gate_pending = []
for source_order, (_, outside_assignment) in enumerate(
    outside_assignments_before_gates.iterrows()
):
    origin_id = outside_assignment["origin_id"]
    if origin_id not in origin_snap_by_id.index:
        raise RuntimeError(
            f"Outside assignment {outside_assignment['assignment_id']} has no origin road snap."
        )

    routes = routes_to_safe_exit_gates(origin_snap_by_id.loc[origin_id])
    if not routes:
        raise RuntimeError(
            f"No selected safe gate is reachable from origin {origin_id}."
        )

    safe_gate_pending.append({
        "source_order": source_order,
        "source": outside_assignment,
        "routes": routes,
        "remaining_people": float(outside_assignment["flooded_people"]),
        "part_number": 0,
    })

safe_gate_remaining_capacity = dict(
    zip(
        safe_exit_gates["gate_id"],
        safe_exit_gates["planning_capacity_people"],
    )
)
safe_gate_allocation_records = []
safe_gate_route_specs = {}

while True:
    active_sources = [
        pending
        for pending in safe_gate_pending
        if pending["remaining_people"] > EPSILON
    ]
    if not active_sources:
        break

    source_candidates = []
    for pending in active_sources:
        viable_routes = [
            route
            for route in pending["routes"]
            if safe_gate_remaining_capacity[route["gate_id"]] > EPSILON
        ]
        if not viable_routes:
            raise RuntimeError(
                "No remaining safe-gate capacity is reachable for "
                f"origin {pending['source']['origin_id']}."
            )

        best_route = viable_routes[0]
        regret_m = (
            viable_routes[1]["route_distance_m"] - best_route["route_distance_m"]
            if len(viable_routes) > 1
            else float("inf")
        )
        source_candidates.append((pending, best_route, regret_m))

    selected_pending, selected_route, _ = sorted(
        source_candidates,
        key=lambda item: (
            -item[2],
            -item[0]["remaining_people"],
            str(item[0]["source"]["origin_id"]),
            item[0]["source_order"],
        ),
    )[0]

    gate_id = selected_route["gate_id"]
    assigned_people = min(
        selected_pending["remaining_people"],
        safe_gate_remaining_capacity[gate_id],
    )
    if assigned_people <= EPSILON:
        raise RuntimeError("Safe-gate allocation made no progress.")

    source = selected_pending["source"]
    gate = safe_gate_by_id.loc[gate_id]
    assignment_id = (
        f"{source['assignment_id']}_{gate_id}_{selected_pending['part_number']:02d}"
    )
    source_rank = pd.to_numeric(source.get("assignment_rank", 0), errors="coerce")
    source_rank = 0 if pd.isna(source_rank) else int(source_rank)

    record = source.drop(labels="geometry").to_dict()
    record.update({
        "geometry": source.geometry,
        "assignment_id": assignment_id,
        "flooded_people": assigned_people,
        # Kept for compatibility with the existing Phase 2 export cells.
        "chosen_destination_type": "outside_flood_area",
        "chosen_destination_id": gate_id,
        "destination_subtype": "deep_safe_egress_gate",
        "safe_exit_gate_id": gate_id,
        "safe_exit_gate_safety_distance_m": gate["safety_distance_m"],
        "safe_exit_gate_east_offset_m": gate["east_offset_m"],
        "chosen_distance_m": selected_route["route_distance_m"],
        "distance_to_outside_m": selected_route["route_distance_m"],
        "network_path_distance_m": selected_route["network_distance_m"],
        "origin_network_access_m": selected_route["origin_access_m"],
        "destination_network_access_m": 0.0,
        "origin_road_segment_m": selected_route["origin_road_segment_m"],
        "destination_road_segment_m": 0.0,
        "origin_network_node_id": selected_route["origin_network_node_id"],
        "destination_network_node_id": selected_route["destination_network_node_id"],
        "origin_road_edge_id": selected_route["origin_edge_id"],
        "destination_road_edge_id": np.nan,
        "assigned_to_building_people": 0.0,
        "assigned_to_outside_people": assigned_people,
        "assignment_rank": source_rank + selected_pending["part_number"],
        "destination_capacity_people": gate["planning_capacity_people"],
        "remaining_capacity_after_assignment": (
            safe_gate_remaining_capacity[gate_id] - assigned_people
        ),
        "refuge_suitability_class": None,
        "access_probability": np.nan,
    })
    safe_gate_allocation_records.append(record)

    safe_gate_route_specs[assignment_id] = {
        "route_kind": "outside",
        "origin_point": source.geometry,
        "origin_snapped_road_point": selected_route["origin_snapped_road_point"],
        "origin_edge_id": selected_route["origin_edge_id"],
        "origin_projection_m": selected_route["origin_projection_m"],
        "origin_endpoint_projection_m": selected_route[
            "origin_endpoint_projection_m"
        ],
        "node_path": selected_route["node_path"],
        "destination_point": selected_route["destination_point"],
        "direct_same_edge": False,
    }

    safe_gate_remaining_capacity[gate_id] -= assigned_people
    selected_pending["remaining_people"] -= assigned_people
    selected_pending["part_number"] += 1

safe_gate_assignments = gpd.GeoDataFrame(
    safe_gate_allocation_records,
    geometry="geometry",
    crs=assignments.crs,
)
assignments = gpd.GeoDataFrame(
    pd.concat(
        [assignments.loc[~outside_mask_before_gates], safe_gate_assignments],
        ignore_index=True,
        sort=False,
    ),
    geometry="geometry",
    crs=safe_gate_assignments.crs,
)
assignment_route_specs.update(safe_gate_route_specs)

outside_people_after_gates = float(
    assignments.loc[
        assignments["safe_exit_gate_id"].notna(),
        "flooded_people",
    ].sum()
)
if not np.isclose(outside_people_before_gates, outside_people_after_gates, atol=EPSILON):
    raise RuntimeError("Safe-gate allocation did not conserve outside demand.")
if not np.isclose(
    float(assignments["flooded_people"].sum()),
    float(demand_origins["flooded_people"].sum()),
    atol=EPSILON,
):
    raise RuntimeError("Safe-gate allocation did not conserve total demand.")

safe_gate_stats = safe_gate_assignments.groupby("safe_exit_gate_id").agg(
    assigned_people=("flooded_people", "sum"),
    assignment_parts=("assignment_id", "size"),
    mean_route_distance_m=("chosen_distance_m", "mean"),
    max_route_distance_m=("chosen_distance_m", "max"),
)
for column in safe_gate_stats.columns:
    safe_exit_gates[column] = safe_exit_gates["gate_id"].map(
        safe_gate_stats[column]
    ).fillna(0.0)
safe_exit_gates["remaining_planning_capacity_people"] = (
    safe_exit_gates["planning_capacity_people"]
    - safe_exit_gates["assigned_people"]
)

if (
    safe_exit_gates["assigned_people"]
    > safe_exit_gates["planning_capacity_people"] + EPSILON
).any():
    raise RuntimeError("A safe gate exceeded its planning capacity.")

# One row per gate rather than one exit point per demand origin. This is used
# by the map and Phase 2 centroid-mapping/export cells.
outside_exit_points = safe_exit_gates.copy()
outside_exit_points["chosen_destination_type"] = "outside_flood_area"
outside_exit_points["chosen_destination_id"] = outside_exit_points["gate_id"]
outside_exit_points["destination_network_node_id"] = outside_exit_points[
    "network_node_id"
]

summary_updates = {
    "assignment_method": (
        "capacity-constrained refuge assignment + deep safe-gate allocation"
    ),
    "assignment_destination_note": (
        "Eight fixed gates at least 1 km outside and east of the flood boundary; "
        "4,000-person planning cap per gate"
    ),
    "assignment_records": len(assignments),
    "people_assigned_to_outside": round(outside_people_after_gates, 2),
    "total_people_assigned": round(float(assignments["flooded_people"].sum()), 2),
    "mean_chosen_network_distance_m": round(
        float(assignments["chosen_distance_m"].mean()), 2
    ),
}
for metric, value in summary_updates.items():
    assignment_summary.loc[assignment_summary["metric"] == metric, "value"] = value

safe_exit_gate_summary = pd.DataFrame([
    ("safe_exit_gate_count", len(safe_exit_gates)),
    (
        "safe_exit_gate_min_hazard_distance_m",
        round(float(safe_exit_gates["safety_distance_m"].min()), 1),
    ),
    (
        "safe_exit_gate_min_east_offset_m",
        round(float(safe_exit_gates["east_offset_m"].min()), 1),
    ),
    (
        "safe_exit_gate_planning_capacity_per_gate",
        SAFE_EXIT_GATE_PLANNING_CAPACITY_PEOPLE,
    ),
    ("safe_exit_gate_total_planning_capacity", total_gate_planning_capacity),
    ("safe_exit_gate_assigned_people", round(outside_people_after_gates, 2)),
    (
        "safe_exit_gate_unused_planning_capacity",
        round(total_gate_planning_capacity - outside_people_after_gates, 2),
    ),
    (
        "safe_exit_gate_max_load_people",
        round(float(safe_exit_gates["assigned_people"].max()), 2),
    ),
    (
        "safe_exit_gate_mean_route_distance_m",
        round(float(safe_gate_assignments["chosen_distance_m"].mean()), 2),
    ),
    (
        "safe_exit_gate_max_route_distance_m",
        round(float(safe_gate_assignments["chosen_distance_m"].max()), 2),
    ),
    (
        "safe_exit_gate_route_note",
        "Approximate undirected network screening; validate routes, directional controls, and capacities in Aimsun",
    ),
], columns=["metric", "value"])
assignment_summary = pd.concat(
    [assignment_summary, safe_exit_gate_summary],
    ignore_index=True,
)

display(safe_exit_gate_summary)
display(
    safe_exit_gates[[
        "gate_id",
        "network_node_id",
        "safety_distance_m",
        "east_offset_m",
        "network_node_degree",
        "planning_capacity_people",
        "assigned_people",
        "remaining_planning_capacity_people",
        "mean_route_distance_m",
    ]]
)


In [ ]:
# ==========================================================
# Map the selected deep safe exit gates
# ==========================================================
safe_gate_map_margin_m = 1_500.0
hazard_minx, hazard_miny, hazard_maxx, hazard_maxy = network_hazard_union.bounds
gate_minx, gate_miny, gate_maxx, gate_maxy = safe_exit_gates.total_bounds
map_minx = min(hazard_minx, gate_minx) - safe_gate_map_margin_m
map_miny = min(hazard_miny, gate_miny) - safe_gate_map_margin_m
map_maxx = max(hazard_maxx, gate_maxx) + safe_gate_map_margin_m
map_maxy = max(hazard_maxy, gate_maxy) + safe_gate_map_margin_m

safe_gate_roads = network_edges_gdf.cx[map_minx:map_maxx, map_miny:map_maxy]

fig, ax = plt.subplots(figsize=(11, 12))
safe_gate_roads.plot(ax=ax, color="#cbd5e1", linewidth=0.35, alpha=0.8, label="road network")
hazard_area.plot(ax=ax, color="#ef4444", alpha=0.22, edgecolor="#991b1b", linewidth=1.0, label="inundation area")
safe_exit_gates.plot(ax=ax, color="#15803d", marker="X", markersize=115, edgecolor="white", linewidth=0.7, label="deep safe exit gate")

for gate in safe_exit_gates.itertuples():
    ax.annotate(
        f"{gate.gate_id}\n{gate.assigned_people:,.0f} people",
        xy=(gate.geometry.x, gate.geometry.y),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=8,
        color="#14532d",
        bbox={"boxstyle": "round,pad=0.18", "fc": "white", "ec": "#86efac", "alpha": 0.9},
    )

ax.set_xlim(map_minx, map_maxx)
ax.set_ylim(map_miny, map_maxy)
ax.set_aspect("equal", adjustable="box")
ax.set_title("Deep Safe Exit Gates: Planned Evacuation Load")
ax.set_xlabel("ITM X")
ax.set_ylabel("ITM Y")
ax.grid(alpha=0.2)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
building_assigned = (
    assignments[assignments["chosen_destination_type"] == "vertical_refuge_building"]
    .groupby("chosen_destination_id")
    .agg(
        assigned_people=("flooded_people", "sum"),
        assigned_origins=("origin_id", "nunique"),
        mean_assignment_distance_m=("chosen_distance_m", "mean"),
    )
    .reset_index()
    .rename(columns={"chosen_destination_id": "destination_id"})
)

building_loads = refuge_destinations.merge(building_assigned, on="destination_id", how="left")
building_loads["assigned_people"] = building_loads["assigned_people"].fillna(0.0).astype(float)
building_loads["assigned_origins"] = building_loads["assigned_origins"].fillna(0).astype("int64")
if "assignment_capacity_people" not in building_loads.columns:
    building_loads["assignment_capacity_people"] = building_loads["capacity_people"]

building_loads["load_ratio"] = building_loads["assigned_people"] / building_loads["assignment_capacity_people"].replace(0, np.nan)
building_loads["load_status"] = pd.cut(
    building_loads["load_ratio"].fillna(0),
    bins=[-0.001, 0.0, 0.8, 1.0, np.inf],
    labels=["unused", "enough_capacity", "near_capacity", "overloaded"],
).astype(str)

building_load_summary = (
    building_loads.groupby("load_status")
    .agg(
        buildings=("destination_id", "size"),
        assigned_people=("assigned_people", "sum"),
        total_assignment_capacity_people=("assignment_capacity_people", "sum"),
        total_theoretical_capacity_people=("capacity_people", "sum"),
    )
    .reset_index()
)

display(pd.DataFrame({"assignment_destination_note": [assignment_destination_note], "eligible_refuge_destinations": [len(refuge_destinations)]}))
display(building_load_summary)
display(building_loads[[
    "destination_id", "id_binyan", "refuge_suitability_class", "capacity_people",
    "assignment_capacity_people", "assigned_people", "assigned_origins", "load_ratio", "load_status"
]].sort_values("assigned_people", ascending=False).head(20))


In [ ]:
# Build geometries from the actual shortest road-network paths.
if "assignment_route_specs" not in globals():
    raise RuntimeError(
        "Run the road-network assignment cell before creating route geometries."
    )

assignment_lines = assignments.copy()
route_geometries = []
missing_route_ids = []

for row in assignments.itertuples():
    route_spec = assignment_route_specs.get(row.assignment_id)

    if route_spec is None:
        missing_route_ids.append(row.assignment_id)
        continue

    route_geometries.append(
        route_geometry_from_spec(route_spec)
    )

if missing_route_ids:
    raise RuntimeError(
        "Route geometry is missing for assignment IDs: "
        + ", ".join(map(str, missing_route_ids[:10]))
    )

assignment_lines["geometry"] = route_geometries
assignment_lines = gpd.GeoDataFrame(
    assignment_lines,
    geometry="geometry",
    crs=assignments.crs,
)

layers_to_write = {
    "flooded_taz_demand": demand_polygons,
    "demand_origin_assignments": assignments,
    "building_loads": building_loads,
    "outside_exit_points": outside_exit_points,
    "assignment_lines": assignment_lines,
}

assignment_gpkg_written = assignment_gpkg_path
assignment_gpkg_status = "written"
try:
    if assignment_gpkg_written.exists():
        assignment_gpkg_written.unlink()
    for layer_name, layer in layers_to_write.items():
        layer.to_file(
            assignment_gpkg_written,
            layer=layer_name,
            driver="GPKG",
        )
except PermissionError:
    assignment_gpkg_written = assignment_gpkg_path.with_name(
        assignment_gpkg_path.stem
        + "_notebook"
        + assignment_gpkg_path.suffix
    )
    if assignment_gpkg_written.exists():
        assignment_gpkg_written.unlink()
    for layer_name, layer in layers_to_write.items():
        layer.to_file(
            assignment_gpkg_written,
            layer=layer_name,
            driver="GPKG",
        )
    assignment_gpkg_status = (
        "fallback_written_because_default_file_was_locked"
    )

assignment_summary_csv_written, assignment_summary_csv_status = (
    write_csv_with_fallback(
        assignment_summary,
        assignment_summary_csv_path,
    )
)

assignment_outputs = pd.DataFrame([
    ("assignment_gpkg", assignment_gpkg_written, assignment_gpkg_status),
    (
        "assignment_summary_csv",
        assignment_summary_csv_written,
        assignment_summary_csv_status,
    ),
], columns=["output", "path", "status"])

display(assignment_outputs)


In [ ]:
building_loads_ta = gpd.clip(building_loads, tel_aviv_window_gdf)
assignments_ta = assignments[assignments.geometry.within(tel_aviv_window)].copy()
assignment_lines_ta = gpd.clip(assignment_lines, tel_aviv_window_gdf)
outside_exit_points_ta = outside_exit_points[outside_exit_points.geometry.within(tel_aviv_window)].copy()

load_colors = {
    "unused": "#cbd5e1",
    "enough_capacity": "#22c55e",
    "near_capacity": "#f59e0b",
    "overloaded": "#ef4444",
}
line_colors = {
    "vertical_refuge_building": "#2563eb",
    "outside_flood_area": "#16a34a",
}

fig, ax = plt.subplots(figsize=(11, 12))
ax.set_facecolor("lightblue")

taz_ta.plot(ax=ax, facecolor="whitesmoke", edgecolor="lightgray", linewidth=0.3, alpha=1.0)
roads_ta.plot(ax=ax, linewidth=0.2, color="gray", alpha=0.35)
hazard_area_ta.plot(ax=ax, color="red", alpha=0.16, edgecolor="darkred", linewidth=0.8)

for destination_type, group in assignment_lines_ta.groupby("chosen_destination_type"):
    group.plot(
        ax=ax,
        color=line_colors.get(destination_type, "#64748b"),
        linewidth=0.7,
        alpha=0.55,
        label=f"assignment to {destination_type}",
    )

for load_status, group in building_loads_ta.groupby("load_status"):
    group.plot(
        ax=ax,
        color=load_colors.get(load_status, "#64748b"),
        edgecolor="black",
        linewidth=0.25,
        alpha=0.9,
        label=f"building: {load_status}",
    )

if len(outside_exit_points_ta) > 0:
    outside_exit_points_ta.plot(
        ax=ax,
        color="#16a34a",
        marker="x",
        markersize=35,
        linewidth=1.0,
        label="deep safe exit gate",
    )

assignments_ta.plot(ax=ax, color="#111827", markersize=12, alpha=0.8, label="demand origin")

ax.set_title("Evacuation Demand Assignment: Refuge Building or Deep Safe Exit Gate")
ax.set_xlabel("ITM X")
ax.set_ylabel("ITM Y")
ax.set_xlim(xmin_view, xmax_view)
ax.set_ylim(ymin_view, ymax_view)
ax.axis("equal")
ax.legend(loc="upper right", fontsize=8)

fig.savefig(assignment_map_path, dpi=300, bbox_inches="tight")
plt.show()

display(pd.DataFrame({"assignment_map_path": [assignment_map_path], "mapped_origins": [len(assignments_ta)]}))


## Phase 2 — Person-Level Baseline Evacuation Scenario

These cells build a deterministic evacuation plan at the alert time: one state and location per person, an assigned destination, a baseline mode, and a planned departure time.


In [ ]:
# ==========================================================
# Phase 2 — baseline scenario configuration and safeguards
# ==========================================================
from pathlib import Path
import csv
import hashlib

BASELINE_SCENARIO_ID = "baseline_mode_continuation_proxy_v1"
BASELINE_PERSON_STATE_SEED = "tsunami_baseline_20260820"
RECOMPUTE_BASELINE_PERSON_SNAPSHOT = False
BASELINE_SNAPSHOT_CHUNK_SIZE = 1_000_000
BASELINE_TIME_BIN_MIN = 5
BASELINE_WALK_SPEED_M_PER_MIN = 75.0
BASELINE_VEHICLE_OCCUPANCY_PROXY = 1.0

# This is a transparent proxy, not a proof of vehicle availability or ownership.
BASELINE_VEHICLE_SOURCE_MODES = {
    "Car",
    "BEV Car",
    "Car Sharing 2",
    "Car Sharing 3",
    "Motorcycle",
}
BASELINE_RESPONSE_DELAY_BY_STATE_MIN = {
    "at_activity": 5.0,
    "at_initial_location": 5.0,
    "in_transit": 10.0,
}

baseline_required_variables = [
    "DEMAND_FILE",
    "ALERT_TIME_HOUR",
    "TSUNAMI_ARRIVAL_HOUR",
    "OUTPUT_DIR",
    "taz_demand",
    "demand_origins",
    "assignments",
    "refuge_destinations",
    "refuge_destination_points",
    "outside_exit_points",
    "write_csv_with_fallback",
]
baseline_missing_variables = [
    name for name in baseline_required_variables if name not in globals()
]
if baseline_missing_variables:
    raise RuntimeError(
        "Run cells 44–51 before Phase 2. Missing variables: "
        + ", ".join(baseline_missing_variables)
    )

BASELINE_OUTPUT_DIR = Path(OUTPUT_DIR)
BASELINE_PERSON_SNAPSHOT_PATH = (
    BASELINE_OUTPUT_DIR / "baseline_person_alert_snapshot.parquet"
)
BASELINE_PLAN_PATH = BASELINE_OUTPUT_DIR / "baseline_evacuation_plan.csv"
BASELINE_PEDESTRIAN_OD_PATH = BASELINE_OUTPUT_DIR / "baseline_pedestrian_od.csv"
BASELINE_VEHICLE_OD_PATH = BASELINE_OUTPUT_DIR / "baseline_vehicle_od.csv"
BASELINE_VEHICLE_TRIPS_PATH = BASELINE_OUTPUT_DIR / "baseline_vehicle_trips.csv"
BASELINE_CLEARANCE_PATH = BASELINE_OUTPUT_DIR / "baseline_traffic_clearance_plan.csv"
BASELINE_CENTROID_MAPPING_PATH = (
    BASELINE_OUTPUT_DIR / "aimsun_centroid_mapping_template.csv"
)
BASELINE_VALIDATION_PATH = BASELINE_OUTPUT_DIR / "baseline_evacuation_validation.csv"
BASELINE_MANIFEST_PATH = BASELINE_OUTPUT_DIR / "baseline_export_manifest.csv"


def baseline_normalize_zone(values):
    """Return stable string zone keys while preserving non-numeric identifiers."""
    numeric = pd.to_numeric(values, errors="coerce")
    numeric_key = numeric.round().astype("Int64").astype("string")
    raw_key = values.astype("string").str.strip()
    return numeric_key.where(numeric_key.notna(), raw_key)


def baseline_stable_hash(value, salt=BASELINE_PERSON_STATE_SEED):
    """Stable hash: the same person receives the same deterministic draw."""
    text_value = f"{salt}|{value}".encode("utf-8")
    return int(hashlib.blake2b(text_value, digest_size=8).hexdigest(), 16)


def baseline_largest_remainder_quotas(weights, total, tie_breakers=None):
    """Allocate an integer total proportionally, conserving the total exactly."""
    weights = pd.Series(weights, dtype="float64").fillna(0.0).clip(lower=0.0)
    total = int(total)

    if total < 0:
        raise ValueError("The requested integer total cannot be negative.")
    if total == 0:
        return pd.Series(0, index=weights.index, dtype="int64")
    if weights.sum() <= 0:
        raise ValueError("Positive allocation requested with zero total weight.")

    raw = weights / weights.sum() * total
    quotas = np.floor(raw).astype("int64")
    remaining = total - int(quotas.sum())

    if remaining:
        if tie_breakers is None:
            ties = pd.Series(weights.index.astype(str), index=weights.index)
        elif isinstance(tie_breakers, pd.Series):
            ties = tie_breakers.reindex(weights.index).astype(str)
        else:
            ties = pd.Series(tie_breakers, index=weights.index).astype(str)

        allocation_order = (
            pd.DataFrame(
                {
                    "fraction": raw - quotas,
                    "tie": ties,
                },
                index=weights.index,
            )
            .sort_values(
                ["fraction", "tie"],
                ascending=[False, True],
                kind="stable",
            )
            .index[:remaining]
        )
        quotas.loc[allocation_order] += 1

    if int(quotas.sum()) != total:
        raise RuntimeError("Largest-remainder allocation did not conserve demand.")
    return quotas.astype("int64")


def baseline_write_parquet_with_fallback(frame, preferred_path):
    """Avoid overwriting a Parquet file if it is open in another program."""
    preferred_path = Path(preferred_path)
    try:
        frame.to_parquet(preferred_path, index=False)
        return preferred_path
    except PermissionError:
        fallback_path = preferred_path.with_name(
            f"{preferred_path.stem}_{pd.Timestamp.now():%Y%m%d_%H%M%S}"
            f"{preferred_path.suffix}"
        )
        frame.to_parquet(fallback_path, index=False)
        return fallback_path


# Schema check: the DAS file has an extra leading raw field in its data rows.
# Deliberately use pandas' default index inference; do NOT add index_col=False,
# which would shift the named fields one column to the left.
baseline_required_das_columns = [
    "person_id",
    "tour_no",
    "stop_no",
    "stop_type",
    "stop_mode",
    "arrival_time",
    "departure_time",
    "prev_stop_departure_time",
    "stop_zone",
    "prev_stop_zone",
    "id",
    "drivetrain",
    "vehicle_drivetrain",
]

with open(DEMAND_FILE, "r", encoding="utf-8-sig", newline="") as baseline_file:
    baseline_reader = csv.reader(baseline_file)
    baseline_header = next(baseline_reader)
    baseline_first_data_row = next(baseline_reader)

baseline_missing_das_columns = sorted(
    set(baseline_required_das_columns) - set(baseline_header)
)
if baseline_missing_das_columns:
    raise RuntimeError(
        "DAS schema is missing required columns: "
        + ", ".join(baseline_missing_das_columns)
    )

baseline_schema_probe = pd.read_csv(
    DEMAND_FILE,
    usecols=baseline_required_das_columns,
    nrows=25,
    low_memory=False,
)
if baseline_schema_probe["person_id"].isna().any():
    raise RuntimeError(
        "DAS person_id could not be read reliably; stop before creating a plan."
    )

baseline_schema_summary = pd.DataFrame(
    [
        ("scenario_id", BASELINE_SCENARIO_ID),
        ("DAS_header_fields", len(baseline_header)),
        ("DAS_first_data_fields", len(baseline_first_data_row)),
        ("pandas_default_reader_used", True),
        ("alert_time_hour", ALERT_TIME_HOUR),
        ("tsunami_arrival_hour", TSUNAMI_ARRIVAL_HOUR),
        ("walk_speed_m_per_min", BASELINE_WALK_SPEED_M_PER_MIN),
        ("vehicle_mode_rule", "DAS stop_mode proxy; no ownership inference"),
    ],
    columns=["metric", "value"],
)
display(baseline_schema_summary)


In [ ]:
# ==========================================================
# Build a one-person, one-location alert snapshot from the DAS
# ==========================================================
baseline_zone_exposure = taz_demand[
    ["TAZV41", "flooded_people"]
].copy()
baseline_zone_exposure["alert_taz"] = baseline_normalize_zone(
    baseline_zone_exposure["TAZV41"]
)
baseline_zone_exposure["flooded_people"] = pd.to_numeric(
    baseline_zone_exposure["flooded_people"],
    errors="coerce",
).fillna(0.0)
baseline_zone_exposure = (
    baseline_zone_exposure[
        baseline_zone_exposure["flooded_people"] > 0
    ]
    .groupby("alert_taz", as_index=False)["flooded_people"]
    .sum()
)
baseline_hazard_zone_keys = set(baseline_zone_exposure["alert_taz"])

if not baseline_hazard_zone_keys:
    raise RuntimeError("No flooded demand zones are available for the baseline.")

baseline_snapshot_columns = [
    "person_id",
    "alert_state",
    "alert_taz",
    "source_stop_mode",
    "source_stop_type",
    "tour_no",
    "stop_no",
    "arrival_time",
    "departure_time",
    "prev_stop_departure_time",
    "prev_stop_zone",
    "stop_zone",
    "das_raw_id",
    "drivetrain",
    "vehicle_drivetrain",
    "transit_progress_to_destination",
    "transit_location_draw",
    "state_location_selection",
    "alert_time_hour",
]

if (
    BASELINE_PERSON_SNAPSHOT_PATH.exists()
    and not RECOMPUTE_BASELINE_PERSON_SNAPSHOT
):
    baseline_person_alert_snapshot = pd.read_parquet(
        BASELINE_PERSON_SNAPSHOT_PATH
    )
    baseline_snapshot_source = "loaded_from_parquet_cache"
else:
    baseline_activity_records = []
    baseline_transit_records = []
    baseline_first_record_candidates = []
    baseline_rows_read = 0

    baseline_numeric_columns = [
        "tour_no",
        "stop_no",
        "arrival_time",
        "departure_time",
        "prev_stop_departure_time",
    ]
    baseline_first_columns = [
        "person_id",
        "tour_no",
        "stop_no",
        "stop_type",
        "stop_mode",
        "arrival_time",
        "departure_time",
        "prev_stop_departure_time",
        "prev_stop_zone",
        "stop_zone",
        "id",
        "drivetrain",
        "vehicle_drivetrain",
    ]

    for baseline_chunk in pd.read_csv(
        DEMAND_FILE,
        usecols=baseline_required_das_columns,
        chunksize=BASELINE_SNAPSHOT_CHUNK_SIZE,
        low_memory=False,
    ):
        baseline_rows_read += len(baseline_chunk)

        for baseline_column in baseline_numeric_columns:
            baseline_chunk[baseline_column] = pd.to_numeric(
                baseline_chunk[baseline_column],
                errors="coerce",
            )

        baseline_chunk["stop_zone_key"] = baseline_normalize_zone(
            baseline_chunk["stop_zone"]
        )
        baseline_chunk["prev_stop_zone_key"] = baseline_normalize_zone(
            baseline_chunk["prev_stop_zone"]
        )

        baseline_activity_mask = (
            (baseline_chunk["arrival_time"] <= ALERT_TIME_HOUR)
            & (baseline_chunk["departure_time"] > ALERT_TIME_HOUR)
            & baseline_chunk["stop_zone_key"].isin(baseline_hazard_zone_keys)
        )
        baseline_activity_records.append(
            baseline_chunk.loc[
                baseline_activity_mask,
                baseline_required_das_columns
                + ["stop_zone_key", "prev_stop_zone_key"],
            ].copy()
        )

        baseline_transit_mask = (
            (baseline_chunk["prev_stop_departure_time"] <= ALERT_TIME_HOUR)
            & (baseline_chunk["arrival_time"] > ALERT_TIME_HOUR)
            & (
                baseline_chunk["prev_stop_zone_key"].isin(
                    baseline_hazard_zone_keys
                )
                | baseline_chunk["stop_zone_key"].isin(
                    baseline_hazard_zone_keys
                )
            )
        )
        baseline_transit_records.append(
            baseline_chunk.loc[
                baseline_transit_mask,
                baseline_required_das_columns
                + ["stop_zone_key", "prev_stop_zone_key"],
            ].copy()
        )

        # A person can cross a chunk boundary. Keep each chunk's earliest
        # candidate and resolve the true first record after concatenation.
        baseline_first_record_candidates.append(
            baseline_chunk.sort_values(
                ["person_id", "prev_stop_departure_time", "tour_no", "stop_no"]
            )
            .groupby("person_id", as_index=False, sort=False)
            .first()[baseline_first_columns]
        )

        print(f"baseline rows read: {baseline_rows_read:,}")

    baseline_at_activity = (
        pd.concat(baseline_activity_records, ignore_index=True)
        .sort_values(["person_id", "tour_no", "stop_no"], kind="stable")
        .drop_duplicates("person_id", keep="first")
    )
    baseline_in_transit = (
        pd.concat(baseline_transit_records, ignore_index=True)
        .sort_values(
            ["person_id", "prev_stop_departure_time", "arrival_time"],
            kind="stable",
        )
        .drop_duplicates("person_id", keep="first")
    )
    baseline_first_records = (
        pd.concat(baseline_first_record_candidates, ignore_index=True)
        .sort_values(
            ["person_id", "prev_stop_departure_time", "tour_no", "stop_no"],
            kind="stable",
        )
        .drop_duplicates("person_id", keep="first")
    )
    baseline_first_records["prev_stop_zone_key"] = baseline_normalize_zone(
        baseline_first_records["prev_stop_zone"]
    )
    baseline_first_records["stop_zone_key"] = baseline_normalize_zone(
        baseline_first_records["stop_zone"]
    )

    # If a malformed record appears in more than one state, preserve the
    # activity state, then transit, then the initial location.
    baseline_activity_ids = set(
        baseline_at_activity["person_id"].dropna().astype(str)
    )
    baseline_in_transit["person_id"] = baseline_in_transit[
        "person_id"
    ].astype(str)
    baseline_in_transit = baseline_in_transit[
        ~baseline_in_transit["person_id"].isin(baseline_activity_ids)
    ].copy()
    baseline_transit_ids = set(
        baseline_in_transit["person_id"].dropna().astype(str)
    )

    baseline_at_activity["person_id"] = baseline_at_activity[
        "person_id"
    ].astype(str)
    baseline_at_initial_location = baseline_first_records[
        (baseline_first_records["prev_stop_departure_time"] > ALERT_TIME_HOUR)
        & baseline_first_records["prev_stop_zone_key"].isin(
            baseline_hazard_zone_keys
        )
        & ~baseline_first_records["person_id"].astype(str).isin(
            baseline_activity_ids | baseline_transit_ids
        )
    ].copy()
    baseline_at_initial_location["person_id"] = (
        baseline_at_initial_location["person_id"].astype(str)
    )

    baseline_transit_duration = (
        baseline_in_transit["arrival_time"]
        - baseline_in_transit["prev_stop_departure_time"]
    )
    baseline_in_transit["transit_progress_to_destination"] = np.where(
        baseline_transit_duration > 0,
        (
            (ALERT_TIME_HOUR - baseline_in_transit[
                "prev_stop_departure_time"
            ])
            / baseline_transit_duration
        ).clip(0, 1),
        0.5,
    )
    baseline_in_transit["transit_location_draw"] = (
        baseline_in_transit["person_id"]
        .map(
            lambda person_id: (
                baseline_stable_hash(person_id, "transit_location") % 1_000_000
            )
            / 1_000_000
        )
        .astype(float)
    )
    baseline_in_transit["state_location_selection"] = np.where(
        baseline_in_transit["transit_location_draw"]
        < baseline_in_transit["transit_progress_to_destination"],
        "estimated_destination_endpoint",
        "estimated_origin_endpoint",
    )
    baseline_in_transit["alert_taz"] = np.where(
        baseline_in_transit["state_location_selection"]
        == "estimated_destination_endpoint",
        baseline_in_transit["stop_zone_key"],
        baseline_in_transit["prev_stop_zone_key"],
    )
    baseline_in_transit = baseline_in_transit[
        baseline_in_transit["alert_taz"].isin(baseline_hazard_zone_keys)
    ].copy()

    def baseline_state_frame(
        frame,
        state,
        zone_column,
        selection,
        transit_progress=np.nan,
        transit_draw=np.nan,
    ):
        result = pd.DataFrame(
            {
                "person_id": frame["person_id"].astype(str),
                "alert_state": state,
                "alert_taz": frame[zone_column].astype("string"),
                "source_stop_mode": frame["stop_mode"].astype("string"),
                "source_stop_type": frame["stop_type"].astype("string"),
                "tour_no": frame["tour_no"],
                "stop_no": frame["stop_no"],
                "arrival_time": frame["arrival_time"],
                "departure_time": frame["departure_time"],
                "prev_stop_departure_time": frame[
                    "prev_stop_departure_time"
                ],
                "prev_stop_zone": frame["prev_stop_zone"],
                "stop_zone": frame["stop_zone"],
                "das_raw_id": frame["id"].astype("string"),
                "drivetrain": frame["drivetrain"].astype("string"),
                "vehicle_drivetrain": frame[
                    "vehicle_drivetrain"
                ].astype("string"),
                "transit_progress_to_destination": transit_progress,
                "transit_location_draw": transit_draw,
                "state_location_selection": selection,
                "alert_time_hour": ALERT_TIME_HOUR,
            }
        )
        return result[baseline_snapshot_columns]

    baseline_snapshot_frames = [
        baseline_state_frame(
            baseline_at_activity,
            "at_activity",
            "stop_zone_key",
            "observed_activity_stop",
        ),
        baseline_state_frame(
            baseline_at_initial_location,
            "at_initial_location",
            "prev_stop_zone_key",
            "initial_record_previous_stop",
        ),
        baseline_state_frame(
            baseline_in_transit,
            "in_transit",
            "alert_taz",
            baseline_in_transit["state_location_selection"],
            baseline_in_transit["transit_progress_to_destination"],
            baseline_in_transit["transit_location_draw"],
        ),
    ]
    baseline_person_alert_snapshot = pd.concat(
        baseline_snapshot_frames,
        ignore_index=True,
    )
    baseline_snapshot_source = "computed_from_raw_DAS"

    if baseline_person_alert_snapshot["person_id"].duplicated().any():
        raise RuntimeError(
            "A person received more than one alert state; baseline plan stopped."
        )

    baseline_person_alert_snapshot = baseline_person_alert_snapshot[
        baseline_person_alert_snapshot["alert_taz"].isin(
            baseline_hazard_zone_keys
        )
    ].copy()
    baseline_person_alert_snapshot = baseline_person_alert_snapshot.sort_values(
        ["alert_taz", "person_id"],
        kind="stable",
    ).reset_index(drop=True)

    BASELINE_PERSON_SNAPSHOT_PATH = baseline_write_parquet_with_fallback(
        baseline_person_alert_snapshot,
        BASELINE_PERSON_SNAPSHOT_PATH,
    )

baseline_person_alert_snapshot["person_id"] = (
    baseline_person_alert_snapshot["person_id"].astype(str)
)
baseline_person_alert_snapshot["alert_taz"] = baseline_normalize_zone(
    baseline_person_alert_snapshot["alert_taz"]
)
baseline_person_alert_snapshot = baseline_person_alert_snapshot[
    baseline_person_alert_snapshot["alert_taz"].isin(
        baseline_hazard_zone_keys
    )
].copy()

if baseline_person_alert_snapshot["person_id"].duplicated().any():
    raise RuntimeError("Cached snapshot contains duplicate person IDs.")

baseline_snapshot_summary = (
    baseline_person_alert_snapshot.groupby("alert_state", as_index=False)
    .agg(
        people=("person_id", "nunique"),
        zones=("alert_taz", "nunique"),
    )
    .sort_values("alert_state")
)
baseline_snapshot_summary = pd.concat(
    [
        pd.DataFrame(
            [
                ("snapshot_source", baseline_snapshot_source, np.nan),
                (
                    "people_in_hazard_snapshot",
                    int(len(baseline_person_alert_snapshot)),
                    int(
                        baseline_person_alert_snapshot[
                            "alert_taz"
                        ].nunique()
                    ),
                ),
                (
                    "fractional_demand_from_phase_1",
                    round(
                        float(baseline_zone_exposure["flooded_people"].sum()),
                        2,
                    ),
                    int(len(baseline_zone_exposure)),
                ),
            ],
            columns=["alert_state", "people", "zones"],
        ),
        baseline_snapshot_summary,
    ],
    ignore_index=True,
)
display(baseline_snapshot_summary)


In [ ]:
# ==========================================================
# Sample people into the flooded share and derive route quotas
# ==========================================================
# The snapshot contains all people represented in every intersecting TAZ.
# Here, flooded_people determines how many of those people are sampled into
# the inundated share. This preserves the Phase 1 population total exactly.
def baseline_capped_quota_allocation(
    weights,
    capacities,
    total,
    tie_breakers,
):
    """Largest-remainder quotas with per-zone candidate limits."""
    weights = pd.Series(weights, dtype="float64").fillna(0.0).clip(lower=0.0)
    capacities = pd.Series(
        np.floor(
            pd.Series(
                capacities,
                index=weights.index,
                dtype="float64",
            )
            .fillna(0.0)
            .clip(lower=0.0)
            .to_numpy()
        ),
        index=weights.index,
        dtype="int64",
    )
    total = int(total)

    if total > int(capacities.sum()):
        raise RuntimeError(
            "There are not enough unique snapshot candidates for the "
            "requested flooded population."
        )

    ideal = weights / weights.sum() * total
    quotas = baseline_largest_remainder_quotas(
        weights,
        total,
        tie_breakers=tie_breakers,
    )
    quotas = pd.Series(
        np.minimum(quotas.to_numpy(), capacities.to_numpy()),
        index=weights.index,
        dtype="int64",
    )

    while int(quotas.sum()) < total:
        eligible = capacities > quotas
        if not eligible.any():
            raise RuntimeError("No eligible candidate remains for redistribution.")

        remaining_order = pd.DataFrame(
            {
                "priority": (ideal - quotas).where(eligible, -np.inf),
                "tie": pd.Series(
                    tie_breakers,
                    index=weights.index,
                ).astype(str),
            },
            index=weights.index,
        ).sort_values(
            ["priority", "tie"],
            ascending=[False, True],
            kind="stable",
        )
        next_index = remaining_order.index[0]
        quotas.loc[next_index] += 1

    if int(quotas.sum()) != total or (quotas > capacities).any():
        raise RuntimeError("Capped demand allocation did not conserve demand.")
    return quotas.astype("int64")


baseline_snapshot_candidate_counts = (
    baseline_person_alert_snapshot.groupby("alert_taz")["person_id"]
    .nunique()
    .rename("available_snapshot_people")
    .reset_index()
)
baseline_zone_integer_quotas = baseline_zone_exposure.merge(
    baseline_snapshot_candidate_counts,
    on="alert_taz",
    how="left",
)
baseline_zone_integer_quotas["available_snapshot_people"] = (
    baseline_zone_integer_quotas["available_snapshot_people"]
    .fillna(0)
    .astype(int)
)
baseline_total_people = int(
    round(float(baseline_zone_exposure["flooded_people"].sum()))
)
baseline_zone_integer_quotas["initial_quota"] = (
    baseline_largest_remainder_quotas(
        baseline_zone_integer_quotas["flooded_people"],
        baseline_total_people,
        tie_breakers=baseline_zone_integer_quotas["alert_taz"],
    )
)
baseline_zone_integer_quotas["candidate_shortfall_before_reallocation"] = (
    baseline_zone_integer_quotas["initial_quota"]
    - baseline_zone_integer_quotas["available_snapshot_people"]
)
baseline_zone_integer_quotas["integer_people"] = (
    baseline_capped_quota_allocation(
        baseline_zone_integer_quotas["flooded_people"],
        baseline_zone_integer_quotas["available_snapshot_people"],
        baseline_total_people,
        baseline_zone_integer_quotas["alert_taz"],
    )
)
baseline_zone_integer_quotas["quota_reallocation_people"] = (
    baseline_zone_integer_quotas["integer_people"]
    - baseline_zone_integer_quotas["initial_quota"]
)
if (
    baseline_zone_integer_quotas["integer_people"]
    > baseline_zone_integer_quotas["available_snapshot_people"]
).any():
    raise RuntimeError("A zone quota exceeds its available unique candidates.")

baseline_assignment_columns = [
    "assignment_id",
    "origin_id",
    "TAZV41",
    "flooded_people",
    "chosen_destination_type",
    "chosen_destination_id",
    "chosen_distance_m",
    "network_path_distance_m",
    "origin_network_access_m",
    "destination_network_access_m",
    "origin_network_node_id",
    "destination_network_node_id",
    "destination_capacity_people",
    "refuge_suitability_class",
    "access_probability",
]
baseline_missing_assignment_columns = [
    column
    for column in baseline_assignment_columns
    if column not in assignments.columns
]
if baseline_missing_assignment_columns:
    raise RuntimeError(
        "The existing assignment table is missing: "
        + ", ".join(baseline_missing_assignment_columns)
    )

baseline_assignment_components = assignments[
    baseline_assignment_columns
].copy()
baseline_assignment_components["alert_taz"] = baseline_normalize_zone(
    baseline_assignment_components["TAZV41"]
)
baseline_assignment_components["flooded_people"] = pd.to_numeric(
    baseline_assignment_components["flooded_people"],
    errors="coerce",
).fillna(0.0)
baseline_assignment_components = baseline_assignment_components[
    (baseline_assignment_components["flooded_people"] > 0)
    & baseline_assignment_components["alert_taz"].isin(
        set(baseline_zone_integer_quotas["alert_taz"])
    )
].copy()

if baseline_assignment_components["assignment_id"].duplicated().any():
    raise RuntimeError("assignment_id must be unique before person allocation.")

baseline_assignment_zone_totals = (
    baseline_assignment_components.groupby("alert_taz")["flooded_people"]
    .sum()
    .rename("assignment_people")
    .reset_index()
)
baseline_assignment_consistency = baseline_zone_integer_quotas.merge(
    baseline_assignment_zone_totals,
    on="alert_taz",
    how="left",
)
baseline_assignment_consistency["assignment_people"] = (
    baseline_assignment_consistency["assignment_people"].fillna(0.0)
)
if not np.allclose(
    baseline_assignment_consistency["flooded_people"],
    baseline_assignment_consistency["assignment_people"],
    atol=1e-6,
):
    raise RuntimeError(
        "The person-demand zones do not match the existing assignment table. "
        "Re-run the demand origins and capacity-constrained assignment first."
    )

baseline_route_quota_parts = []
for _, baseline_zone_row in baseline_zone_integer_quotas.iterrows():
    baseline_zone_key = baseline_zone_row["alert_taz"]
    baseline_zone_people = int(baseline_zone_row["integer_people"])
    baseline_zone_components = baseline_assignment_components[
        baseline_assignment_components["alert_taz"] == baseline_zone_key
    ].copy()

    baseline_zone_components["baseline_route_quota"] = (
        baseline_largest_remainder_quotas(
            baseline_zone_components["flooded_people"],
            baseline_zone_people,
            tie_breakers=baseline_zone_components["assignment_id"],
        )
    )
    baseline_route_quota_parts.append(baseline_zone_components)

baseline_assignment_quotas = pd.concat(
    baseline_route_quota_parts,
    ignore_index=True,
)
if int(baseline_assignment_quotas["baseline_route_quota"].sum()) != (
    baseline_total_people
):
    raise RuntimeError("Route quotas do not conserve the baseline population.")

baseline_building_integer_loads = (
    baseline_assignment_quotas[
        baseline_assignment_quotas["chosen_destination_type"]
        == "vertical_refuge_building"
    ]
    .groupby("chosen_destination_id")["baseline_route_quota"]
    .sum()
)
baseline_refuge_capacity = (
    refuge_destinations.set_index("destination_id")[
        "assignment_capacity_people"
    ]
)
baseline_capacity_check = pd.DataFrame(
    {
        "integer_assigned_people": baseline_building_integer_loads,
        "assignment_capacity_people": baseline_refuge_capacity.reindex(
            baseline_building_integer_loads.index
        ),
    }
)
baseline_capacity_check["capacity_excess_people"] = (
    baseline_capacity_check["integer_assigned_people"]
    - baseline_capacity_check["assignment_capacity_people"]
)
baseline_overloaded_refuges = baseline_capacity_check[
    baseline_capacity_check["capacity_excess_people"] > 1e-6
]
if len(baseline_overloaded_refuges) > 0:
    raise RuntimeError(
        "Integer allocation would exceed refuge capacity. Revisit the "
        "assignment or rounding policy before using the plan."
    )

baseline_discrete_demand_summary = pd.DataFrame(
    [
        (
            "phase_1_fractional_reference_people",
            round(float(baseline_zone_exposure["flooded_people"].sum()), 2),
        ),
        ("integer_people_planned", baseline_total_people),
        (
            "initial_zone_candidate_shortfall_people",
            int(
                baseline_zone_integer_quotas[
                    "candidate_shortfall_before_reallocation"
                ].clip(lower=0)
                .sum()
            ),
        ),
        (
            "zones_rebalanced_for_unique_people",
            int(
                (
                    baseline_zone_integer_quotas[
                        "quota_reallocation_people"
                    ] != 0
                ).sum()
            ),
        ),
        ("overloaded_refuges_after_rounding", len(baseline_overloaded_refuges)),
    ],
    columns=["metric", "value"],
)
display(baseline_discrete_demand_summary)


In [ ]:
# ==========================================================
# Preserve deep-safe-gate planning caps after integer rounding
# ==========================================================
# The fractional allocation respects every gate cap. This cell moves only
# rounding units, keeping each person's alert TAZ and origin unchanged.
if "safe_exit_gates" not in globals():
    raise RuntimeError("Run the deep safe-gate assignment cell before Phase 2.")

baseline_safe_gate_caps = safe_exit_gates.set_index("gate_id")[
    "planning_capacity_people"
].astype(int)
baseline_safe_gate_ids = set(baseline_safe_gate_caps.index)
baseline_safe_gate_mask = baseline_assignment_quotas[
    "chosen_destination_id"
].isin(baseline_safe_gate_ids)
baseline_zone_quota_before_gate_rebalance = (
    baseline_assignment_quotas.groupby("alert_taz")["baseline_route_quota"]
    .sum()
    .sort_index()
)

baseline_safe_gate_load_before = (
    baseline_assignment_quotas.loc[baseline_safe_gate_mask]
    .groupby("chosen_destination_id")["baseline_route_quota"]
    .sum()
    .reindex(baseline_safe_gate_caps.index, fill_value=0)
    .astype(int)
)
baseline_safe_gate_load_after = baseline_safe_gate_load_before.copy()
baseline_safe_gate_rounding_moves = []

for source_gate_id in sorted(baseline_safe_gate_caps.index):
    while (
        baseline_safe_gate_load_after.loc[source_gate_id]
        > baseline_safe_gate_caps.loc[source_gate_id]
    ):
        source_candidates = baseline_assignment_quotas.index[
            (
                baseline_assignment_quotas["chosen_destination_id"]
                == source_gate_id
            )
            & (
                baseline_assignment_quotas["baseline_route_quota"]
                > np.floor(baseline_assignment_quotas["flooded_people"])
            )
        ].tolist()

        candidate_moves = []
        for source_index in source_candidates:
            source_row = baseline_assignment_quotas.loc[source_index]
            origin_id = source_row["origin_id"]
            gate_routes = {
                route["gate_id"]: route
                for route in routes_to_safe_exit_gates(
                    origin_snap_by_id.loc[origin_id]
                )
            }

            for target_gate_id in baseline_safe_gate_caps.index:
                if (
                    target_gate_id == source_gate_id
                    or baseline_safe_gate_load_after.loc[target_gate_id]
                    >= baseline_safe_gate_caps.loc[target_gate_id]
                    or target_gate_id not in gate_routes
                ):
                    continue

                target_route = gate_routes[target_gate_id]
                candidate_moves.append({
                    "source_index": source_index,
                    "source_gate_id": source_gate_id,
                    "target_gate_id": target_gate_id,
                    "target_route": target_route,
                    "distance_delta_m": (
                        target_route["route_distance_m"]
                        - float(source_row["chosen_distance_m"])
                    ),
                    "source_assignment_id": source_row["assignment_id"],
                    "alert_taz": source_row["alert_taz"],
                    "origin_id": origin_id,
                })

        if not candidate_moves:
            raise RuntimeError(
                "Could not preserve the integer cap for "
                f"{source_gate_id}; no alternative gate route is available."
            )

        selected_move = min(
            candidate_moves,
            key=lambda move: (
                move["distance_delta_m"],
                str(move["source_assignment_id"]),
                move["target_gate_id"],
            ),
        )
        source_index = selected_move["source_index"]
        source_row = baseline_assignment_quotas.loc[source_index].copy()
        target_gate_id = selected_move["target_gate_id"]
        target_gate = safe_exit_gates.set_index("gate_id").loc[target_gate_id]
        target_route = selected_move["target_route"]
        synthetic_assignment_id = (
            f"{source_row['assignment_id']}_INT_{target_gate_id}_{len(baseline_safe_gate_rounding_moves):02d}"
        )

        # Remove one rounded-up person from the full gate, then append a
        # same-origin alternative component for a gate with spare capacity.
        baseline_assignment_quotas.loc[
            source_index, "baseline_route_quota"
        ] -= 1
        synthetic_route = source_row.copy()
        synthetic_route["assignment_id"] = synthetic_assignment_id
        synthetic_route["flooded_people"] = 1.0
        synthetic_route["baseline_route_quota"] = 1
        synthetic_route["chosen_destination_type"] = "outside_flood_area"
        synthetic_route["chosen_destination_id"] = target_gate_id
        synthetic_route["chosen_distance_m"] = target_route["route_distance_m"]
        synthetic_route["network_path_distance_m"] = target_route["network_distance_m"]
        synthetic_route["origin_network_access_m"] = target_route["origin_access_m"]
        synthetic_route["destination_network_access_m"] = 0.0
        synthetic_route["origin_network_node_id"] = target_route["origin_network_node_id"]
        synthetic_route["destination_network_node_id"] = target_route["destination_network_node_id"]
        synthetic_route["destination_capacity_people"] = target_gate["planning_capacity_people"]
        synthetic_route["refuge_suitability_class"] = None
        synthetic_route["access_probability"] = np.nan

        baseline_assignment_quotas = pd.concat(
            [baseline_assignment_quotas, pd.DataFrame([synthetic_route])],
            ignore_index=True,
        )
        baseline_assignment_components = pd.concat(
            [
                baseline_assignment_components,
                pd.DataFrame([synthetic_route]).reindex(
                    columns=baseline_assignment_components.columns
                ),
            ],
            ignore_index=True,
        )
        baseline_safe_gate_load_after.loc[source_gate_id] -= 1
        baseline_safe_gate_load_after.loc[target_gate_id] += 1
        baseline_safe_gate_rounding_moves.append({
            **{
                key: value
                for key, value in selected_move.items()
                if key != "target_route"
            },
            "synthetic_assignment_id": synthetic_assignment_id,
        })

baseline_zone_quota_after_gate_rebalance = (
    baseline_assignment_quotas.groupby("alert_taz")["baseline_route_quota"]
    .sum()
    .sort_index()
)
if not baseline_zone_quota_before_gate_rebalance.equals(
    baseline_zone_quota_after_gate_rebalance
):
    raise RuntimeError("Safe-gate rounding changed a TAZ population total.")
if int(baseline_assignment_quotas["baseline_route_quota"].sum()) != baseline_total_people:
    raise RuntimeError("Safe-gate rounding did not conserve people.")
if (baseline_safe_gate_load_after > baseline_safe_gate_caps).any():
    raise RuntimeError("A deep safe-gate cap is still exceeded after rounding.")

baseline_safe_gate_rounding_moves = pd.DataFrame(
    baseline_safe_gate_rounding_moves
)
baseline_safe_gate_integer_loads = pd.DataFrame({
    "gate_id": baseline_safe_gate_caps.index,
    "planning_capacity_people": baseline_safe_gate_caps.values,
    "load_before_rounding_fix": baseline_safe_gate_load_before.values,
    "load_after_rounding_fix": baseline_safe_gate_load_after.values,
})
baseline_safe_gate_integer_loads["remaining_capacity_after_rounding"] = (
    baseline_safe_gate_integer_loads["planning_capacity_people"]
    - baseline_safe_gate_integer_loads["load_after_rounding_fix"]
)

baseline_discrete_demand_summary = pd.concat([
    baseline_discrete_demand_summary,
    pd.DataFrame([
        ("safe_gate_rounding_reassignments", len(baseline_safe_gate_rounding_moves)),
        ("safe_gate_max_load_after_rounding", int(baseline_safe_gate_load_after.max())),
    ], columns=["metric", "value"]),
], ignore_index=True)

display(baseline_safe_gate_integer_loads)
if len(baseline_safe_gate_rounding_moves) > 0:
    display(baseline_safe_gate_rounding_moves)


In [ ]:
# ==========================================================
# Build one evacuation decision for every sampled person
# ==========================================================
# Select people without replacement within each TAZ. The stable hash makes the
# uniform-within-zone sampling reproducible for this scenario and seed.
baseline_selected_parts = []
for _, baseline_zone_row in baseline_zone_integer_quotas.iterrows():
    baseline_zone_key = baseline_zone_row["alert_taz"]
    baseline_zone_people = int(baseline_zone_row["integer_people"])
    baseline_zone_candidates = baseline_person_alert_snapshot[
        baseline_person_alert_snapshot["alert_taz"] == baseline_zone_key
    ].copy()
    baseline_zone_candidates["baseline_person_hash"] = (
        baseline_zone_candidates["person_id"]
        .map(lambda person_id: baseline_stable_hash(person_id, "person_plan"))
    )
    baseline_zone_candidates = baseline_zone_candidates.sort_values(
        ["baseline_person_hash", "person_id"],
        kind="stable",
    )
    baseline_selected_parts.append(
        baseline_zone_candidates.head(baseline_zone_people)
    )

baseline_selected_people = pd.concat(
    baseline_selected_parts,
    ignore_index=True,
)
if (
    len(baseline_selected_people) != baseline_total_people
    or baseline_selected_people["person_id"].duplicated().any()
):
    raise RuntimeError("The selected person population is not one row per person.")

# Match people to the route components that were already chosen by the
# capacity-constrained road-network assignment. The pairing is deterministic.
baseline_plan_parts = []
for baseline_zone_key, baseline_zone_people in baseline_selected_people.groupby(
    "alert_taz",
    sort=False,
):
    baseline_zone_people = baseline_zone_people.sort_values(
        ["baseline_person_hash", "person_id"],
        kind="stable",
    ).copy()
    baseline_zone_routes = baseline_assignment_quotas[
        (baseline_assignment_quotas["alert_taz"] == baseline_zone_key)
        & (baseline_assignment_quotas["baseline_route_quota"] > 0)
    ].sort_values("assignment_id", kind="stable")

    baseline_assignment_ids = np.repeat(
        baseline_zone_routes["assignment_id"].astype(str).to_numpy(),
        baseline_zone_routes["baseline_route_quota"].astype(int).to_numpy(),
    )
    if len(baseline_assignment_ids) != len(baseline_zone_people):
        raise RuntimeError(
            f"Route allocation mismatch in alert zone {baseline_zone_key}."
        )

    baseline_zone_people["assignment_id"] = baseline_assignment_ids
    baseline_plan_parts.append(baseline_zone_people)

baseline_evacuation_plan = pd.concat(
    baseline_plan_parts,
    ignore_index=True,
).merge(
    baseline_assignment_components.drop(columns=["TAZV41", "flooded_people"]),
    on=["assignment_id", "alert_taz"],
    how="left",
    validate="many_to_one",
)

if baseline_evacuation_plan["chosen_destination_id"].isna().any():
    raise RuntimeError("Some people could not be matched to an assigned target.")

# Transparent baseline mode rule. A vehicle mode is a proxy inferred only
# from DAS stop_mode; it does not claim verified ownership or occupancy.
baseline_evacuation_plan["source_stop_mode"] = (
    baseline_evacuation_plan["source_stop_mode"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)
baseline_evacuation_plan["evacuation_mode"] = np.where(
    baseline_evacuation_plan["source_stop_mode"].isin(
        BASELINE_VEHICLE_SOURCE_MODES
    ),
    "vehicle_proxy",
    "walk",
)
baseline_evacuation_plan["mode_decision_reason"] = np.where(
    baseline_evacuation_plan["evacuation_mode"] == "vehicle_proxy",
    "DAS stop_mode is in the explicit vehicle-proxy set",
    "walk default because vehicle availability is not established",
)
baseline_evacuation_plan["response_delay_min"] = (
    baseline_evacuation_plan["alert_state"]
    .map(BASELINE_RESPONSE_DELAY_BY_STATE_MIN)
    .fillna(5.0)
)
baseline_evacuation_plan["planned_departure_time_s"] = (
    baseline_evacuation_plan["response_delay_min"] * 60
).round().astype(int)
baseline_evacuation_plan["planned_departure_time_hour"] = (
    ALERT_TIME_HOUR
    + baseline_evacuation_plan["response_delay_min"] / 60.0
)

# Walking receives only a transparent free-flow proxy. Vehicle travel time is
# intentionally left unknown until it is simulated in Aimsun.
baseline_evacuation_plan["estimated_walk_travel_time_min"] = np.where(
    baseline_evacuation_plan["evacuation_mode"] == "walk",
    baseline_evacuation_plan["chosen_distance_m"]
    / BASELINE_WALK_SPEED_M_PER_MIN,
    np.nan,
)
baseline_evacuation_plan["estimated_arrival_time_hour"] = (
    baseline_evacuation_plan["planned_departure_time_hour"]
    + baseline_evacuation_plan["estimated_walk_travel_time_min"] / 60.0
)
baseline_evacuation_plan["estimated_arrives_before_tsunami"] = pd.Series(
    pd.NA,
    index=baseline_evacuation_plan.index,
    dtype="boolean",
)
baseline_walk_mask = baseline_evacuation_plan["evacuation_mode"] == "walk"
baseline_evacuation_plan.loc[
    baseline_walk_mask,
    "estimated_arrives_before_tsunami",
] = (
    baseline_evacuation_plan.loc[
        baseline_walk_mask,
        "estimated_arrival_time_hour",
    ]
    <= TSUNAMI_ARRIVAL_HOUR
).to_numpy()

baseline_evacuation_plan["transit_treatment"] = np.where(
    baseline_evacuation_plan["alert_state"] == "in_transit",
    "deterministic endpoint draw based on trip progress",
    "not_in_transit",
)
baseline_evacuation_plan["vehicle_proxy_id"] = np.where(
    baseline_evacuation_plan["evacuation_mode"] == "vehicle_proxy",
    "VP_" + baseline_evacuation_plan["person_id"].astype(str),
    pd.NA,
)
baseline_evacuation_plan["plan_status"] = np.where(
    baseline_evacuation_plan["evacuation_mode"] == "walk",
    "planned_walk_time_is_a_free_flow_proxy",
    "planned_vehicle_requires_Aimsun_simulation",
)
baseline_evacuation_plan["scenario_id"] = BASELINE_SCENARIO_ID
baseline_evacuation_plan = baseline_evacuation_plan.sort_values(
    ["planned_departure_time_s", "person_id"],
    kind="stable",
).reset_index(drop=True)
baseline_evacuation_plan.insert(
    0,
    "plan_id",
    "EP_" + (baseline_evacuation_plan.index + 1).astype(str).str.zfill(6),
)

baseline_plan_summary = pd.DataFrame(
    [
        ("scenario_id", BASELINE_SCENARIO_ID),
        (
            "phase_1_fractional_reference_people",
            round(float(baseline_zone_exposure["flooded_people"].sum()), 2),
        ),
        ("integer_people_planned", int(len(baseline_evacuation_plan))),
        (
            "people_to_vertical_refuge",
            int(
                (
                    baseline_evacuation_plan["chosen_destination_type"]
                    == "vertical_refuge_building"
                ).sum()
            ),
        ),
        (
            "people_to_outside_flood_area",
            int(
                (
                    baseline_evacuation_plan["chosen_destination_type"]
                    == "outside_flood_area"
                ).sum()
            ),
        ),
        (
            "walk_people",
            int(
                (baseline_evacuation_plan["evacuation_mode"] == "walk").sum()
            ),
        ),
        (
            "vehicle_proxy_people",
            int(
                (
                    baseline_evacuation_plan["evacuation_mode"]
                    == "vehicle_proxy"
                ).sum()
            ),
        ),
        (
            "walkers_with_free_flow_arrival_before_tsunami",
            int(
                baseline_evacuation_plan[
                    "estimated_arrives_before_tsunami"
                ].fillna(False).sum()
            ),
        ),
    ],
    columns=["metric", "value"],
)
display(baseline_plan_summary)
display(
    baseline_evacuation_plan[
        [
            "plan_id",
            "person_id",
            "alert_state",
            "alert_taz",
            "evacuation_mode",
            "chosen_destination_type",
            "chosen_destination_id",
            "chosen_distance_m",
            "planned_departure_time_s",
            "plan_status",
        ]
    ].head(15)
)


In [ ]:
# ==========================================================
# Export baseline inputs, QA, and an Aimsun centroid-mapping template
# ==========================================================
baseline_origin_lookup = demand_origins[
    ["origin_id", "geometry"]
].copy()
baseline_origin_lookup["origin_x_itm"] = (
    baseline_origin_lookup.geometry.x
)
baseline_origin_lookup["origin_y_itm"] = (
    baseline_origin_lookup.geometry.y
)
baseline_origin_lookup = pd.DataFrame(
    baseline_origin_lookup.drop(columns="geometry")
).drop_duplicates("origin_id")

baseline_refuge_destination_lookup = refuge_destination_points[
    ["destination_id", "geometry"]
].copy()
baseline_refuge_destination_lookup["destination_x_itm"] = (
    baseline_refuge_destination_lookup.geometry.x
)
baseline_refuge_destination_lookup["destination_y_itm"] = (
    baseline_refuge_destination_lookup.geometry.y
)
baseline_refuge_destination_lookup = pd.DataFrame(
    baseline_refuge_destination_lookup.drop(columns="geometry")
)

baseline_outside_destination_lookup = outside_exit_points[
    ["chosen_destination_id", "geometry"]
].copy().rename(columns={"chosen_destination_id": "destination_id"})
baseline_outside_destination_lookup["destination_x_itm"] = (
    baseline_outside_destination_lookup.geometry.x
)
baseline_outside_destination_lookup["destination_y_itm"] = (
    baseline_outside_destination_lookup.geometry.y
)
baseline_outside_destination_lookup = pd.DataFrame(
    baseline_outside_destination_lookup.drop(columns="geometry")
).drop_duplicates("destination_id")

baseline_destination_lookup = pd.concat(
    [
        baseline_refuge_destination_lookup,
        baseline_outside_destination_lookup,
    ],
    ignore_index=True,
)
if baseline_destination_lookup["destination_id"].duplicated().any():
    raise RuntimeError(
        "Destination IDs must be unique before creating the Aimsun mapping."
    )

baseline_coordinate_columns = [
    "origin_x_itm",
    "origin_y_itm",
    "destination_x_itm",
    "destination_y_itm",
]
baseline_evacuation_plan = (
    baseline_evacuation_plan.drop(
        columns=baseline_coordinate_columns,
        errors="ignore",
    )
    .merge(
        baseline_origin_lookup,
        on="origin_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        baseline_destination_lookup,
        left_on="chosen_destination_id",
        right_on="destination_id",
        how="left",
        validate="many_to_one",
    )
    .drop(columns="destination_id")
)

baseline_evacuation_plan["departure_time_bin_s"] = (
    baseline_evacuation_plan["planned_departure_time_s"]
    // (BASELINE_TIME_BIN_MIN * 60)
    * (BASELINE_TIME_BIN_MIN * 60)
).astype(int)

baseline_pedestrian_od = (
    baseline_evacuation_plan[
        baseline_evacuation_plan["evacuation_mode"] == "walk"
    ]
    .groupby(
        [
            "scenario_id",
            "departure_time_bin_s",
            "origin_id",
            "chosen_destination_id",
            "chosen_destination_type",
            "origin_x_itm",
            "origin_y_itm",
            "destination_x_itm",
            "destination_y_itm",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        pedestrian_count=("plan_id", "size"),
        mean_route_distance_m=("chosen_distance_m", "mean"),
        mean_response_delay_min=("response_delay_min", "mean"),
    )
    .sort_values(
        ["departure_time_bin_s", "origin_id", "chosen_destination_id"],
        kind="stable",
    )
)

baseline_vehicle_trips = baseline_evacuation_plan[
    baseline_evacuation_plan["evacuation_mode"] == "vehicle_proxy"
].copy()
baseline_vehicle_trips.insert(
    0,
    "vehicle_trip_id",
    "VT_" + baseline_vehicle_trips["plan_id"].astype(str),
)
baseline_vehicle_trips["vehicle_proxy_count"] = (
    1.0 / BASELINE_VEHICLE_OCCUPANCY_PROXY
)
baseline_vehicle_trips["aimsun_vehicle_type"] = (
    "TO_BE_MAPPED_from_DAS_stop_mode"
)

baseline_vehicle_od = (
    baseline_vehicle_trips.groupby(
        [
            "scenario_id",
            "departure_time_bin_s",
            "origin_id",
            "chosen_destination_id",
            "chosen_destination_type",
            "origin_x_itm",
            "origin_y_itm",
            "destination_x_itm",
            "destination_y_itm",
            "source_stop_mode",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        vehicle_count=("vehicle_proxy_count", "sum"),
        represented_people=("plan_id", "size"),
        mean_route_distance_m=("chosen_distance_m", "mean"),
        mean_response_delay_min=("response_delay_min", "mean"),
    )
    .sort_values(
        ["departure_time_bin_s", "origin_id", "chosen_destination_id"],
        kind="stable",
    )
)

# The advanced eastbound / traffic-clearance policy is deliberately staged,
# not silently applied to this baseline scenario.
baseline_traffic_clearance_plan = pd.DataFrame(
    columns=[
        "clearance_id",
        "scenario_id",
        "policy_status",
        "target_population",
        "direction_or_destination_rule",
        "start_time_s",
        "end_time_s",
        "implementation_note",
    ]
)

baseline_aimsun_centroid_mapping = pd.concat(
    [
        baseline_origin_lookup.rename(
            columns={"origin_id": "entity_id"}
        ).assign(
            entity_type="origin",
            x_itm=lambda frame: frame["origin_x_itm"],
            y_itm=lambda frame: frame["origin_y_itm"],
        ),
        baseline_destination_lookup.rename(
            columns={"destination_id": "entity_id"}
        ).assign(
            entity_type="destination",
            x_itm=lambda frame: frame["destination_x_itm"],
            y_itm=lambda frame: frame["destination_y_itm"],
        ),
    ],
    ignore_index=True,
)
baseline_aimsun_centroid_mapping["candidate_aimsun_centroid_id"] = pd.NA
baseline_aimsun_centroid_mapping["mapping_status"] = "to_be_mapped_in_Aimsun"
baseline_aimsun_centroid_mapping["coordinate_reference_system"] = (
    "EPSG:2039 (ITM)"
)
baseline_aimsun_centroid_mapping = baseline_aimsun_centroid_mapping[
    [
        "entity_type",
        "entity_id",
        "x_itm",
        "y_itm",
        "candidate_aimsun_centroid_id",
        "mapping_status",
        "coordinate_reference_system",
    ]
].sort_values(["entity_type", "entity_id"], kind="stable")

baseline_missing_coordinates = int(
    baseline_evacuation_plan[baseline_coordinate_columns].isna().any(axis=1).sum()
)
baseline_validation = pd.DataFrame(
    [
        (
            "one_row_per_person",
            not baseline_evacuation_plan["person_id"].duplicated().any(),
            int(len(baseline_evacuation_plan)),
            "Each selected person has exactly one plan row.",
        ),
        (
            "integer_population_conserved",
            len(baseline_evacuation_plan) == baseline_total_people,
            int(len(baseline_evacuation_plan)),
            "Plan rows equal the largest-remainder demand total.",
        ),
        (
            "route_quotas_conserved",
            int(baseline_assignment_quotas["baseline_route_quota"].sum())
            == baseline_total_people,
            int(baseline_assignment_quotas["baseline_route_quota"].sum()),
            "Person assignments preserve all route quotas.",
        ),
        (
            "refuge_capacity_not_exceeded_after_rounding",
            len(baseline_overloaded_refuges) == 0,
            int(len(baseline_overloaded_refuges)),
            "Checked after converting fractional allocations to people.",
        ),
        (
            "all_plan_rows_have_export_coordinates",
            baseline_missing_coordinates == 0,
            baseline_missing_coordinates,
            "ITM coordinates are supplied for the Aimsun mapping stage.",
        ),
        (
            "pedestrian_od_people",
            int(baseline_pedestrian_od["pedestrian_count"].sum())
            == int((baseline_evacuation_plan["evacuation_mode"] == "walk").sum()),
            int(baseline_pedestrian_od["pedestrian_count"].sum()),
            "Aggregated walking OD demand matches the person plan.",
        ),
        (
            "vehicle_od_people",
            np.isclose(
                baseline_vehicle_od["represented_people"].sum(),
                (baseline_evacuation_plan["evacuation_mode"] == "vehicle_proxy").sum(),
            ),
            int(baseline_vehicle_od["represented_people"].sum()),
            "Vehicle-proxy OD demand matches the person plan.",
        ),
    ],
    columns=["check", "passed", "observed_value", "note"],
)

baseline_written_outputs = []
for baseline_output_name, baseline_output_frame, baseline_output_path in [
    (
        "evacuation_plan",
        baseline_evacuation_plan,
        BASELINE_PLAN_PATH,
    ),
    (
        "pedestrian_od",
        baseline_pedestrian_od,
        BASELINE_PEDESTRIAN_OD_PATH,
    ),
    (
        "vehicle_od",
        baseline_vehicle_od,
        BASELINE_VEHICLE_OD_PATH,
    ),
    (
        "vehicle_trips",
        baseline_vehicle_trips,
        BASELINE_VEHICLE_TRIPS_PATH,
    ),
    (
        "traffic_clearance_plan",
        baseline_traffic_clearance_plan,
        BASELINE_CLEARANCE_PATH,
    ),
    (
        "aimsun_centroid_mapping_template",
        baseline_aimsun_centroid_mapping,
        BASELINE_CENTROID_MAPPING_PATH,
    ),
    (
        "validation",
        baseline_validation,
        BASELINE_VALIDATION_PATH,
    ),
]:
    baseline_written_path, baseline_write_status = write_csv_with_fallback(
        baseline_output_frame,
        baseline_output_path,
    )
    baseline_written_outputs.append(
        {
            "output": baseline_output_name,
            "path": str(baseline_written_path),
            "status": baseline_write_status,
            "rows": int(len(baseline_output_frame)),
        }
    )

baseline_manifest = pd.DataFrame(
    [
        {
            "output": "person_alert_snapshot",
            "path": str(BASELINE_PERSON_SNAPSHOT_PATH),
            "status": baseline_snapshot_source,
            "rows": int(len(baseline_person_alert_snapshot)),
        },
        *baseline_written_outputs,
    ]
)
baseline_manifest_path, baseline_manifest_status = write_csv_with_fallback(
    baseline_manifest,
    BASELINE_MANIFEST_PATH,
)
baseline_manifest = pd.concat(
    [
        baseline_manifest,
        pd.DataFrame(
            [
                {
                    "output": "export_manifest",
                    "path": str(baseline_manifest_path),
                    "status": baseline_manifest_status,
                    "rows": int(len(baseline_manifest)),
                }
            ]
        ),
    ],
    ignore_index=True,
)

if not baseline_validation["passed"].all():
    display(baseline_validation)
    raise RuntimeError(
        "Baseline export validation failed; outputs should not be imported "
        "into Aimsun until the failed checks are resolved."
    )

display(baseline_validation)
display(baseline_manifest)


## Aimsun Centroid and Connector Candidate Mapping

No Aimsun model with centroid and connector IDs is available in this workspace. This section creates geographic attachment candidates and proposed labels; confirm them in Aimsun before assigning actual IDs.


In [ ]:
# ==========================================================
# Build Aimsun centroid and connector attachment candidates
# ==========================================================
from shapely.geometry import LineString

AIMSUN_CONNECTOR_REVIEW_THRESHOLD_M = 100.0
AIMSUN_MAPPING_CANDIDATES_PATH = (
    Path(OUTPUT_DIR) / "aimsun_centroid_attachment_candidates.csv"
)
AIMSUN_MAPPING_QA_PATH = Path(OUTPUT_DIR) / "aimsun_centroid_mapping_qa.csv"
AIMSUN_PEDESTRIAN_OD_CANDIDATES_PATH = (
    Path(OUTPUT_DIR) / "aimsun_pedestrian_od_centroid_candidates.csv"
)
AIMSUN_VEHICLE_OD_CANDIDATES_PATH = (
    Path(OUTPUT_DIR) / "aimsun_vehicle_od_centroid_candidates.csv"
)

aimsun_mapping_required = [
    "baseline_evacuation_plan",
    "baseline_pedestrian_od",
    "baseline_vehicle_od",
    "demand_origins",
    "refuge_destination_points",
    "safe_exit_gates",
    "roads",
    "write_csv_with_fallback",
]
aimsun_mapping_missing = [
    name for name in aimsun_mapping_required if name not in globals()
]
if aimsun_mapping_missing:
    raise RuntimeError(
        "Run the Phase 2 export cell before Aimsun mapping. Missing: "
        + ", ".join(aimsun_mapping_missing)
    )

# Demand totals identify exactly which sources and destinations are active
# in the baseline plan. Unused refuge candidates are intentionally omitted.
aimsun_origin_usage = (
    baseline_evacuation_plan.groupby("origin_id", as_index=False)
    .agg(
        represented_people=("plan_id", "size"),
        pedestrian_people=(
            "evacuation_mode",
            lambda modes: int((modes == "walk").sum()),
        ),
        vehicle_proxy_people=(
            "evacuation_mode",
            lambda modes: int((modes == "vehicle_proxy").sum()),
        ),
    )
)
aimsun_destination_usage = (
    baseline_evacuation_plan.groupby(
        ["chosen_destination_id", "chosen_destination_type"],
        as_index=False,
    )
    .agg(
        represented_people=("plan_id", "size"),
        pedestrian_people=(
            "evacuation_mode",
            lambda modes: int((modes == "walk").sum()),
        ),
        vehicle_proxy_people=(
            "evacuation_mode",
            lambda modes: int((modes == "vehicle_proxy").sum()),
        ),
    )
)

aimsun_origin_entities = demand_origins[["origin_id", "geometry"]].drop_duplicates(
    "origin_id"
).merge(
    aimsun_origin_usage,
    on="origin_id",
    how="inner",
    validate="one_to_one",
)
aimsun_origin_entities = gpd.GeoDataFrame(
    aimsun_origin_entities,
    geometry="geometry",
    crs=demand_origins.crs,
)
aimsun_origin_entities = aimsun_origin_entities.rename(
    columns={"origin_id": "entity_id"}
)
aimsun_origin_entities["entity_role"] = "origin"

aimsun_safe_gate_ids = set(safe_exit_gates["gate_id"])
aimsun_refuge_usage = aimsun_destination_usage[
    aimsun_destination_usage["chosen_destination_type"]
    == "vertical_refuge_building"
].copy()
aimsun_refuge_entities = aimsun_refuge_usage.merge(
    refuge_destination_points[["destination_id", "geometry"]],
    left_on="chosen_destination_id",
    right_on="destination_id",
    how="left",
    validate="one_to_one",
)
if aimsun_refuge_entities["geometry"].isna().any():
    raise RuntimeError("An active refuge destination has no geometry.")
aimsun_refuge_entities = gpd.GeoDataFrame(
    aimsun_refuge_entities,
    geometry="geometry",
    crs=refuge_destination_points.crs,
).rename(columns={"chosen_destination_id": "entity_id"})
aimsun_refuge_entities["entity_role"] = "refuge_destination"

aimsun_gate_usage = aimsun_destination_usage[
    aimsun_destination_usage["chosen_destination_id"].isin(
        aimsun_safe_gate_ids
    )
].copy()
aimsun_gate_entities = aimsun_gate_usage.merge(
    safe_exit_gates[["gate_id", "geometry"]],
    left_on="chosen_destination_id",
    right_on="gate_id",
    how="left",
    validate="one_to_one",
)
if aimsun_gate_entities["geometry"].isna().any():
    raise RuntimeError("An active safe gate has no geometry.")
aimsun_gate_entities = gpd.GeoDataFrame(
    aimsun_gate_entities,
    geometry="geometry",
    crs=safe_exit_gates.crs,
).rename(columns={"chosen_destination_id": "entity_id"})
aimsun_gate_entities["entity_role"] = "safe_exit_gate"

aimsun_entity_columns = [
    "entity_role",
    "entity_id",
    "represented_people",
    "pedestrian_people",
    "vehicle_proxy_people",
    "geometry",
]
aimsun_mapping_entities = gpd.GeoDataFrame(
    pd.concat([
        aimsun_origin_entities[aimsun_entity_columns],
        aimsun_refuge_entities[aimsun_entity_columns],
        aimsun_gate_entities[aimsun_entity_columns],
    ], ignore_index=True),
    geometry="geometry",
    crs=demand_origins.crs,
)
aimsun_mapping_entities["entity_id"] = aimsun_mapping_entities[
    "entity_id"
].astype(str)
aimsun_mapping_entities["mapping_key"] = (
    aimsun_mapping_entities["entity_role"]
    + ":"
    + aimsun_mapping_entities["entity_id"]
)
if aimsun_mapping_entities["mapping_key"].duplicated().any():
    raise RuntimeError("An active Aimsun mapping entity is duplicated.")

aimsun_label_prefix = {
    "origin": "EV_ORIGIN_",
    "refuge_destination": "EV_REFUGE_",
    "safe_exit_gate": "EV_GATE_",
}
aimsun_mapping_entities["proposed_centroid_label"] = (
    aimsun_mapping_entities["entity_role"].map(aimsun_label_prefix)
    + aimsun_mapping_entities["entity_id"]
)
aimsun_mapping_entities["x_itm"] = aimsun_mapping_entities.geometry.x
aimsun_mapping_entities["y_itm"] = aimsun_mapping_entities.geometry.y

# `roads` is the supplied Aimsun-source road geometry. The section fields
# below are attachment candidates, not confirmed IDs in an absent .ang model.
aimsun_sections = roads[["id", "eid", "geometry"]].copy().rename(
    columns={
        "id": "candidate_source_section_id",
        "eid": "candidate_source_section_eid",
    }
)
if aimsun_sections.crs != aimsun_mapping_entities.crs:
    aimsun_sections = aimsun_sections.to_crs(aimsun_mapping_entities.crs)
aimsun_section_geometry_by_index = aimsun_sections.geometry.to_dict()

aimsun_mapping_candidates = gpd.sjoin_nearest(
    aimsun_mapping_entities,
    aimsun_sections,
    how="left",
    distance_col="nearest_source_section_distance_m",
)
aimsun_mapping_candidates = (
    aimsun_mapping_candidates.sort_values(
        ["nearest_source_section_distance_m", "mapping_key"],
        kind="stable",
    )
    .drop_duplicates("mapping_key", keep="first")
    .reset_index(drop=True)
)

aimsun_attachment_points = []
aimsun_connector_wkts = []
for _, entity in aimsun_mapping_candidates.iterrows():
    section_geometry = aimsun_section_geometry_by_index.get(
        entity.get("index_right")
    )
    if section_geometry is None or section_geometry.is_empty:
        aimsun_attachment_points.append(None)
        aimsun_connector_wkts.append(None)
        continue

    attachment_point = section_geometry.interpolate(
        section_geometry.project(entity.geometry)
    )
    aimsun_attachment_points.append(attachment_point)
    aimsun_connector_wkts.append(
        LineString([entity.geometry, attachment_point]).wkt
    )

aimsun_mapping_candidates["attachment_point"] = aimsun_attachment_points
aimsun_mapping_candidates["attachment_x_itm"] = [
    point.x if point is not None else np.nan
    for point in aimsun_attachment_points
]
aimsun_mapping_candidates["attachment_y_itm"] = [
    point.y if point is not None else np.nan
    for point in aimsun_attachment_points
]
aimsun_mapping_candidates["connector_wkt"] = aimsun_connector_wkts
aimsun_mapping_candidates["candidate_aimsun_centroid_id"] = pd.NA
aimsun_mapping_candidates["candidate_aimsun_connector_id"] = pd.NA
aimsun_mapping_candidates["mapping_method"] = (
    "nearest supplied source-road section; confirm in the Aimsun model"
)
aimsun_mapping_candidates["mapping_status"] = np.select(
    [
        aimsun_mapping_candidates["nearest_source_section_distance_m"].isna(),
        aimsun_mapping_candidates["nearest_source_section_distance_m"]
        <= 25.0,
        aimsun_mapping_candidates["nearest_source_section_distance_m"]
        <= AIMSUN_CONNECTOR_REVIEW_THRESHOLD_M,
    ],
    [
        "unmapped_no_source_section",
        "candidate_ready_for_Aimsun_review",
        "review_connector_25_to_100m",
    ],
    default="manual_review_connector_over_100m",
)
aimsun_mapping_candidates["coordinate_reference_system"] = "EPSG:2039 (ITM)"

aimsun_mapping_output_columns = [
    "mapping_key",
    "entity_role",
    "entity_id",
    "proposed_centroid_label",
    "represented_people",
    "pedestrian_people",
    "vehicle_proxy_people",
    "x_itm",
    "y_itm",
    "candidate_source_section_id",
    "candidate_source_section_eid",
    "attachment_x_itm",
    "attachment_y_itm",
    "nearest_source_section_distance_m",
    "connector_wkt",
    "candidate_aimsun_centroid_id",
    "candidate_aimsun_connector_id",
    "mapping_method",
    "mapping_status",
    "coordinate_reference_system",
]
aimsun_mapping_candidates = pd.DataFrame(
    aimsun_mapping_candidates[aimsun_mapping_output_columns]
).sort_values(["entity_role", "entity_id"], kind="stable")

aimsun_origin_labels = aimsun_mapping_candidates[
    aimsun_mapping_candidates["entity_role"] == "origin"
][["entity_id", "proposed_centroid_label"]].rename(
    columns={
        "entity_id": "origin_id",
        "proposed_centroid_label": "origin_centroid_label",
    }
)
aimsun_destination_labels = aimsun_mapping_candidates[
    aimsun_mapping_candidates["entity_role"] != "origin"
][["entity_id", "proposed_centroid_label"]].rename(
    columns={
        "entity_id": "chosen_destination_id",
        "proposed_centroid_label": "destination_centroid_label",
    }
)
if aimsun_destination_labels["chosen_destination_id"].duplicated().any():
    raise RuntimeError("A destination has more than one proposed centroid.")

aimsun_pedestrian_od_candidates = (
    baseline_pedestrian_od.merge(
        aimsun_origin_labels,
        on="origin_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        aimsun_destination_labels,
        on="chosen_destination_id",
        how="left",
        validate="many_to_one",
    )
)
aimsun_vehicle_od_candidates = (
    baseline_vehicle_od.merge(
        aimsun_origin_labels,
        on="origin_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        aimsun_destination_labels,
        on="chosen_destination_id",
        how="left",
        validate="many_to_one",
    )
)

aimsun_od_label_missing = int(
    aimsun_pedestrian_od_candidates[[
        "origin_centroid_label", "destination_centroid_label"
    ]].isna().any(axis=1).sum()
    + aimsun_vehicle_od_candidates[[
        "origin_centroid_label", "destination_centroid_label"
    ]].isna().any(axis=1).sum()
)
if aimsun_od_label_missing > 0:
    raise RuntimeError("At least one OD row has no proposed centroid label.")

aimsun_mapping_qa = pd.DataFrame([
    (
        "active_mapping_entities",
        len(aimsun_mapping_candidates),
        "Active origins plus used refuge and safe-gate destinations.",
    ),
    (
        "entities_requiring_manual_connector_review",
        int((
            aimsun_mapping_candidates["nearest_source_section_distance_m"]
            > AIMSUN_CONNECTOR_REVIEW_THRESHOLD_M
        ).sum()),
        f"Distance exceeds {AIMSUN_CONNECTOR_REVIEW_THRESHOLD_M:.0f} m.",
    ),
    (
        "maximum_candidate_connector_distance_m",
        round(float(aimsun_mapping_candidates["nearest_source_section_distance_m"].max()), 2),
        "Use the candidate only after visual confirmation in Aimsun.",
    ),
    (
        "pedestrian_OD_rows_with_proposed_labels",
        len(aimsun_pedestrian_od_candidates),
        "Every row has origin and destination labels.",
    ),
    (
        "vehicle_OD_rows_with_proposed_labels",
        len(aimsun_vehicle_od_candidates),
        "Every row has origin and destination labels.",
    ),
    (
        "actual_Aimsun_centroid_ids_assigned",
        0,
        "Expected until the correct Aimsun .ang model is available.",
    ),
], columns=["check", "observed_value", "note"])

aimsun_mapping_written_outputs = []
for aimsun_output_name, aimsun_output_frame, aimsun_output_path in [
    (
        "centroid_attachment_candidates",
        aimsun_mapping_candidates,
        AIMSUN_MAPPING_CANDIDATES_PATH,
    ),
    ("centroid_mapping_qa", aimsun_mapping_qa, AIMSUN_MAPPING_QA_PATH),
    (
        "pedestrian_od_centroid_candidates",
        aimsun_pedestrian_od_candidates,
        AIMSUN_PEDESTRIAN_OD_CANDIDATES_PATH,
    ),
    (
        "vehicle_od_centroid_candidates",
        aimsun_vehicle_od_candidates,
        AIMSUN_VEHICLE_OD_CANDIDATES_PATH,
    ),
]:
    aimsun_written_path, aimsun_write_status = write_csv_with_fallback(
        aimsun_output_frame,
        aimsun_output_path,
    )
    aimsun_mapping_written_outputs.append({
        "output": aimsun_output_name,
        "path": str(aimsun_written_path),
        "status": aimsun_write_status,
        "rows": len(aimsun_output_frame),
    })

display(aimsun_mapping_qa)
display(aimsun_mapping_candidates.head(12))
display(pd.DataFrame(aimsun_mapping_written_outputs))


## Phase 3 — Aimsun OD Import Readiness

This section creates a protected mapping-completion template. Once actual Aimsun centroid IDs are entered, it generates vehicle and pedestrian OD files with `from_centroid_id`, `to_centroid_id`, and time intervals.


In [ ]:
# ==========================================================
# Phase 3 — Aimsun OD import readiness
# ==========================================================
AIMSUN_COMPLETED_MAPPING_PATH = (
    Path(OUTPUT_DIR) / "aimsun_centroid_id_mapping_completed.csv"
)
AIMSUN_IMPORT_READINESS_PATH = (
    Path(OUTPUT_DIR) / "aimsun_import_readiness.csv"
)
AIMSUN_VEHICLE_OD_READY_PATH = (
    Path(OUTPUT_DIR) / "aimsun_vehicle_od_ready.csv"
)
AIMSUN_PEDESTRIAN_OD_READY_PATH = (
    Path(OUTPUT_DIR) / "aimsun_pedestrian_od_ready.csv"
)

phase3_required = [
    "aimsun_mapping_candidates",
    "baseline_vehicle_od",
    "baseline_pedestrian_od",
    "write_csv_with_fallback",
    "BASELINE_TIME_BIN_MIN",
]
phase3_missing = [name for name in phase3_required if name not in globals()]
if phase3_missing:
    raise RuntimeError(
        "Run the Aimsun candidate-mapping cell before this cell. Missing: "
        + ", ".join(phase3_missing)
    )

phase3_mapping_template = aimsun_mapping_candidates[[
    "mapping_key",
    "entity_role",
    "entity_id",
    "proposed_centroid_label",
    "candidate_source_section_id",
    "candidate_source_section_eid",
    "attachment_x_itm",
    "attachment_y_itm",
    "nearest_source_section_distance_m",
    "mapping_status",
]].copy()
phase3_mapping_template["aimsun_centroid_id"] = pd.NA
phase3_mapping_template["aimsun_connector_id"] = pd.NA
phase3_mapping_template["model_confirmation_note"] = (
    "Fill after creating or matching the centroid and connector in Aimsun"
)

phase3_written_outputs = []
if not AIMSUN_COMPLETED_MAPPING_PATH.exists():
    phase3_template_path, phase3_template_status = write_csv_with_fallback(
        phase3_mapping_template,
        AIMSUN_COMPLETED_MAPPING_PATH,
    )
    phase3_written_outputs.append({
        "output": "centroid_id_mapping_template",
        "path": str(phase3_template_path),
        "status": phase3_template_status,
        "rows": len(phase3_mapping_template),
    })

phase3_completed_mapping_exists = AIMSUN_COMPLETED_MAPPING_PATH.exists()
phase3_mapping_complete = False
phase3_missing_centroid_ids = len(phase3_mapping_template)
aimsun_vehicle_od_ready = None
aimsun_pedestrian_od_ready = None

if phase3_completed_mapping_exists:
    phase3_completed_mapping = pd.read_csv(AIMSUN_COMPLETED_MAPPING_PATH)
    phase3_required_mapping_columns = [
        "mapping_key",
        "proposed_centroid_label",
        "aimsun_centroid_id",
    ]
    phase3_missing_mapping_columns = [
        column
        for column in phase3_required_mapping_columns
        if column not in phase3_completed_mapping.columns
    ]
    if phase3_missing_mapping_columns:
        raise RuntimeError(
            "The completed mapping file is missing columns: "
            + ", ".join(phase3_missing_mapping_columns)
        )

    phase3_completed_mapping["mapping_key"] = (
        phase3_completed_mapping["mapping_key"].astype(str).str.strip()
    )
    phase3_completed_mapping["aimsun_centroid_id"] = (
        phase3_completed_mapping["aimsun_centroid_id"]
        .astype("string")
        .str.strip()
    )
    phase3_completed_mapping.loc[
        phase3_completed_mapping["aimsun_centroid_id"].isin(["", "<NA>", "nan"]),
        "aimsun_centroid_id",
    ] = pd.NA

    phase3_expected_keys = set(phase3_mapping_template["mapping_key"])
    phase3_provided_keys = set(phase3_completed_mapping["mapping_key"])
    phase3_unknown_keys = sorted(phase3_provided_keys - phase3_expected_keys)
    if phase3_unknown_keys:
        raise RuntimeError(
            "The completed mapping has unknown keys: "
            + ", ".join(phase3_unknown_keys[:10])
        )
    if phase3_completed_mapping["mapping_key"].duplicated().any():
        raise RuntimeError("The completed mapping contains duplicate mapping keys.")

    phase3_completed_mapping = phase3_mapping_template[[
        "mapping_key", "entity_role", "entity_id", "proposed_centroid_label"
    ]].merge(
        phase3_completed_mapping[["mapping_key", "aimsun_centroid_id"]],
        on="mapping_key",
        how="left",
        validate="one_to_one",
    )
    phase3_missing_centroid_ids = int(
        phase3_completed_mapping["aimsun_centroid_id"].isna().sum()
    )
    phase3_mapping_complete = phase3_missing_centroid_ids == 0

    if phase3_mapping_complete:
        if phase3_completed_mapping["aimsun_centroid_id"].duplicated().any():
            raise RuntimeError(
                "Each project entity must have its own Aimsun centroid ID."
            )

        phase3_origin_lookup = phase3_completed_mapping[
            phase3_completed_mapping["entity_role"] == "origin"
        ][["entity_id", "aimsun_centroid_id"]].rename(
            columns={
                "entity_id": "origin_id",
                "aimsun_centroid_id": "from_centroid_id",
            }
        )
        phase3_destination_lookup = phase3_completed_mapping[
            phase3_completed_mapping["entity_role"] != "origin"
        ][["entity_id", "aimsun_centroid_id"]].rename(
            columns={
                "entity_id": "chosen_destination_id",
                "aimsun_centroid_id": "to_centroid_id",
            }
        )

        def phase3_make_ready_od(od_frame, demand_column):
            ready = od_frame.copy()
            ready["origin_id"] = ready["origin_id"].astype(str)
            ready["chosen_destination_id"] = (
                ready["chosen_destination_id"].astype(str)
            )
            ready = (
                ready.merge(
                    phase3_origin_lookup,
                    on="origin_id",
                    how="left",
                    validate="many_to_one",
                )
                .merge(
                    phase3_destination_lookup,
                    on="chosen_destination_id",
                    how="left",
                    validate="many_to_one",
                )
            )
            if ready[["from_centroid_id", "to_centroid_id"]].isna().any(axis=1).any():
                raise RuntimeError("An OD row has no completed centroid mapping.")
            ready["start_time_s"] = ready["departure_time_bin_s"].astype(int)
            ready["end_time_s"] = (
                ready["start_time_s"] + BASELINE_TIME_BIN_MIN * 60
            )
            ready["demand_measure"] = demand_column
            ready["import_status"] = (
                "ready_for_model_specific_Aimsun_import"
            )
            return ready

        aimsun_vehicle_od_ready = phase3_make_ready_od(
            baseline_vehicle_od, "vehicle_count"
        )
        aimsun_pedestrian_od_ready = phase3_make_ready_od(
            baseline_pedestrian_od, "pedestrian_count"
        )

        for phase3_output_name, phase3_output_frame, phase3_output_path in [
            (
                "vehicle_od_ready",
                aimsun_vehicle_od_ready,
                AIMSUN_VEHICLE_OD_READY_PATH,
            ),
            (
                "pedestrian_od_ready",
                aimsun_pedestrian_od_ready,
                AIMSUN_PEDESTRIAN_OD_READY_PATH,
            ),
        ]:
            phase3_written_path, phase3_write_status = write_csv_with_fallback(
                phase3_output_frame,
                phase3_output_path,
            )
            phase3_written_outputs.append({
                "output": phase3_output_name,
                "path": str(phase3_written_path),
                "status": phase3_write_status,
                "rows": len(phase3_output_frame),
            })

phase3_model_files = sorted(
    str(path) for path in Path.cwd().rglob("*.ang")
)
phase3_readiness = pd.DataFrame([
    (
        "candidate_mapping_entities",
        len(phase3_mapping_template),
        "One proposed centroid for each active origin and destination.",
    ),
    (
        "completed_centroid_ids",
        len(phase3_mapping_template) - phase3_missing_centroid_ids,
        "Fill actual IDs in aimsun_centroid_id_mapping_completed.csv.",
    ),
    (
        "missing_centroid_ids",
        phase3_missing_centroid_ids,
        "No model-specific OD import is generated until this is zero.",
    ),
    (
        "Aimsun_model_files_found",
        len(phase3_model_files),
        "A .ang model is required for the actual Aimsun import.",
    ),
    (
        "vehicle_OD_ready",
        aimsun_vehicle_od_ready is not None,
        "Contains from/to centroid IDs after mapping completion.",
    ),
    (
        "pedestrian_OD_ready",
        aimsun_pedestrian_od_ready is not None,
        "Requires an Aimsun pedestrian model before operational use.",
    ),
], columns=["check", "observed_value", "note"])

phase3_readiness_path, phase3_readiness_status = write_csv_with_fallback(
    phase3_readiness,
    AIMSUN_IMPORT_READINESS_PATH,
)
phase3_written_outputs.append({
    "output": "import_readiness",
    "path": str(phase3_readiness_path),
    "status": phase3_readiness_status,
    "rows": len(phase3_readiness),
})

display(phase3_readiness)
display(pd.DataFrame(phase3_written_outputs))


## Phase 4 — Evacuation Scenario Variations

Every scenario below starts from a protected copy of the validated baseline plan. The baseline outputs are not overwritten. Scenario files are written under `outputs/scenarios/`; all headings and output fields are in English.

The controlled gate and closure variants use the same approximate road-network routing assumption as the existing assignment. Eastbound access restrictions, road closures, signal control, and background traffic remain documented policies until the actual Aimsun network is available.

In [ ]:
# ==========================================================
# Phase 4 — Build independent evacuation scenario variants
# ==========================================================
import importlib
import scenario_variations
importlib.reload(scenario_variations)
from scenario_variations import build_evacuation_scenarios

phase4_required = [
    "baseline_evacuation_plan",
    "safe_exit_gates",
    "origin_snap_by_id",
    "routes_to_safe_exit_gates",
    "OUTPUT_DIR",
    "ALERT_TIME_HOUR",
    "TSUNAMI_ARRIVAL_HOUR",
    "BASELINE_TIME_BIN_MIN",
    "BASELINE_WALK_SPEED_M_PER_MIN",
    "BASELINE_VEHICLE_OCCUPANCY_PROXY",
    "write_csv_with_fallback",
    "baseline_write_parquet_with_fallback",
]
phase4_missing = [name for name in phase4_required if name not in globals()]
if phase4_missing:
    raise RuntimeError(
        "Run the baseline, road-network, deep-safe-gate, and Phase 2 cells first. Missing: "
        + ", ".join(phase4_missing)
    )

scenario_results = build_evacuation_scenarios(
    baseline_evacuation_plan=baseline_evacuation_plan,
    safe_exit_gates=safe_exit_gates,
    origin_snap_by_id=origin_snap_by_id,
    routes_to_safe_exit_gates=routes_to_safe_exit_gates,
    output_dir=OUTPUT_DIR,
    alert_time_hour=ALERT_TIME_HOUR,
    tsunami_arrival_hour=TSUNAMI_ARRIVAL_HOUR,
    time_bin_min=BASELINE_TIME_BIN_MIN,
    walk_speed_m_per_min=BASELINE_WALK_SPEED_M_PER_MIN,
    vehicle_occupancy_proxy=BASELINE_VEHICLE_OCCUPANCY_PROXY,
    write_csv_with_fallback=write_csv_with_fallback,
    write_parquet_with_fallback=baseline_write_parquet_with_fallback,
    aimsun_mapping_candidates=globals().get("aimsun_mapping_candidates"),
)

scenario_config = scenario_results["scenario_config"]
scenario_summary = scenario_results["scenario_summary"]
scenario_validation = scenario_results["scenario_validation"]
scenario_control_plan = scenario_results["scenario_control_plan"]
scenario_plans = scenario_results["scenario_plans"]
scenario_vehicle_od = scenario_results["scenario_vehicle_od"]
scenario_pedestrian_od = scenario_results["scenario_pedestrian_od"]
scenario_gate_reassignment_audit = scenario_results["scenario_gate_reassignment_audit"]
scenario_export_manifest = scenario_results["scenario_export_manifest"]

if not scenario_validation["passed"].all():
    raise RuntimeError("At least one Phase 4 scenario validation check failed.")

display(scenario_config)
display(scenario_summary)
display(
    scenario_validation.groupby("scenario_id", as_index=False)["passed"]
    .all()
    .rename(columns={"passed": "all_validation_checks_passed"})
)
display(scenario_control_plan)
display(scenario_export_manifest)
